## Combining all the data

In [1]:
# collecting Y prediction datasets
import os
import gzip
import pandas as pd

# Define folder path where the files are stored
folder_path = "/Users/etloaner/Documents/ASU/Capstone project/Storm event dataset"

# Function to read compressed CSV files
def read_gz_csv(file_path):
    with gzip.open(file_path, "rt") as f:
        return pd.read_csv(f, low_memory=False)

# Lists to store data
details_list = []
locations_list = []
fatalities_list = []

# Read each file and append to corresponding list
for file in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file)
    
    if "details" in file:
        df = read_gz_csv(file_path)
        df.columns = df.columns.str.strip().str.lower()  # Standardizing column names
        details_list.append(df)
    elif "locations" in file:
        df = read_gz_csv(file_path)
        df.columns = df.columns.str.strip().str.lower()
        locations_list.append(df)
    elif "fatalities" in file:
        df = read_gz_csv(file_path)
        df.columns = df.columns.str.strip().str.lower()
        fatalities_list.append(df)

# Concatenate all dataframes (append)
details_df = pd.concat(details_list, ignore_index=True) if details_list else pd.DataFrame()
locations_df = pd.concat(locations_list, ignore_index=True) if locations_list else pd.DataFrame()
fatalities_df = pd.concat(fatalities_list, ignore_index=True) if fatalities_list else pd.DataFrame()

# Print column names for debugging
print("Details Columns:", details_df.columns)
print("Locations Columns:", locations_df.columns)
print("Fatalities Columns:", fatalities_df.columns)

# Ensure 'event_id' exists in all DataFrames before merging
if "event_id" not in details_df.columns:
    raise KeyError("event_id missing from details dataset")
if "event_id" not in locations_df.columns:
    raise KeyError("event_id missing from locations dataset")
if "event_id" not in fatalities_df.columns:
    raise KeyError("event_id missing from fatalities dataset")

# Merge data using event_id
merged_df = details_df.merge(locations_df, on="event_id", how="left") \
                      .merge(fatalities_df, on="event_id", how="left")

# Save final dataframe to CSV
output_file = "final_storm_data.csv"
merged_df.to_csv(output_file, index=False)

print(f"Final consolidated dataset saved as {output_file}")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/Users/etloaner/Documents/ASU/Capstone project/Storm event dataset'

## Cleaning and filtering the data

In [ ]:
import pandas as pd
storm_df = pd.read_csv("/Users/etloaner/Documents/ASU/Capstone project/final_storm_data.csv")


import pandas as pd
import numpy as np

# Function to convert damage strings (e.g., "10.00K", "10.00M") to a numeric value
def convert_damage(damage_str):
    try:
        if pd.isnull(damage_str):
            return 0.0
        damage_str = str(damage_str).strip()
        if damage_str.endswith('K'):
            return float(damage_str[:-1]) * 1e3
        elif damage_str.endswith('M'):
            return float(damage_str[:-1]) * 1e6
        else:
            return float(damage_str)
    except Exception:
        return 0.0

# Function to categorize an event as extreme (1) or non-extreme (0)
def categorize_extreme(row):
    # Get the event type and ensure no extra spaces
    event_type = str(row.get('event_type', '')).strip()
    
    # Convert damages to numeric values
    damage_property = convert_damage(row.get('damage_property', '0.00K'))
    damage_crops = convert_damage(row.get('damage_crops', '0.00K'))
    total_damage = damage_property + damage_crops
    
    # Sum injuries and deaths from direct and indirect counts
    total_injuries = (row.get('injuries_direct', 0) or 0) + (row.get('injuries_indirect', 0) or 0)
    total_deaths = (row.get('deaths_direct', 0) or 0) + (row.get('deaths_indirect', 0) or 0)
    
    # Set default thresholds (adjust these based on your domain knowledge)
    damage_threshold = 100000       # Example: $100K damage threshold
    injury_threshold = 20           # Example: 20 or more injuries
    death_threshold = 5             # Example: 5 or more deaths
    wind_speed_threshold = 50       # Wind speed in knots for extreme wind events
    hail_size_threshold = 1.5       # Hail size in inches considered extreme
    
    # Overall impact: if damage, injuries, or deaths exceed thresholds, mark as extreme.
    if total_damage >= damage_threshold or total_injuries >= injury_threshold or total_deaths >= death_threshold:
        return 1
    
    # Specific rules per event type
    if event_type == 'Tornado':
        # Use the Enhanced Fujita Scale: EF2 or higher is considered extreme.
        tor_scale = str(row.get('tor_f_scale', 'EF0')).strip()
        try:
            scale_value = int(tor_scale.replace('EF', ''))
        except Exception:
            scale_value = 0
        if scale_value >= 2:
            return 1

    if event_type in ['Thunderstorm Wind', 'High Wind', 'Strong Wind', 'Marine Thunderstorm Wind']:
        # For wind events, use the magnitude (assumed to be in knots)
        try:
            magnitude = float(row.get('magnitude', 0))
        except Exception:
            magnitude = 0
        if magnitude >= wind_speed_threshold:
            return 1

    if event_type == 'Hail':
        # For hail events, use the magnitude (assumed to be in inches)
        try:
            magnitude = float(row.get('magnitude', 0))
        except Exception:
            magnitude = 0
        if magnitude >= hail_size_threshold:
            return 1

    if event_type in ['Flash Flood', 'Flood', 'Coastal Flood']:
        # Flood events may be extreme if they cause high damage (already checked above) 
        if total_damage >= damage_threshold:
            return 1

    if event_type in ['Heavy Rain', 'Excessive Heat', 'Heat']:
        # These events may be extreme if they result in significant injuries or damage.
        if total_injuries >= injury_threshold or total_damage >= damage_threshold:
            return 1

    if event_type in ['Winter Storm', 'Winter Weather', 'Blizzard']:
        # Winter events are flagged based on impact
        if total_damage >= damage_threshold or total_injuries >= injury_threshold:
            return 1

    # Additional event-specific rules can be added here

    # If none of the conditions are met, mark as non-extreme.
    return 0

# Load your consolidated dataset (ensure that column names are standardized, e.g., lower-case and stripped)
df = pd.read_csv("final_storm_data.csv")
df.columns = df.columns.str.strip().str.lower()  # e.g., converting "Event_Type" to "event_type"

# Apply the categorization function to each row in the DataFrame
df['extreme'] = df.apply(categorize_extreme, axis=1)

# Save the updated DataFrame with the extreme flag to a new CSV file
output_file = "final_storm_data_with_extreme_flag.csv"
df.to_csv(output_file, index=False)

print(f"Categorization complete. Saved final file as '{output_file}'.")


/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_15249/405993759.py:2: DtypeWarning: Columns (16,25,26,28,29,34,35,37,39,40,42,43,48,49,55,56,68,69) have mixed types. Specify dtype option on import or set low_memory=False.
  storm_df = pd.read_csv("/Users/etloaner/Documents/ASU/Capstone project/final_storm_data.csv")
/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_15249/405993759.py:98: DtypeWarning: Columns (16,25,26,28,29,34,35,37,39,40,42,43,48,49,55,56,68,69) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("final_storm_data.csv")


Categorization complete. Saved final file as 'final_storm_data_with_extreme_flag.csv'.


## Getting only last few years data

In [ ]:
import pandas as pd
final_extreme_event_dataset = pd.read_csv("/Users/etloaner/Documents/ASU/Capstone project/final_storm_data_with_extreme_flag.csv")
filtered_df = final_extreme_event_dataset[final_extreme_event_dataset["year"].isin([2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020])]
filtered_df.drop(columns=["episode_id_x","event_id","state_fips","cz_type","cz_fips","cz_name","wfo","source","tor_other_wfo","tor_other_cz_state","tor_other_cz_fips","tor_other_cz_name","episode_narrative","event_narrative","data_source","episode_id_y","azimuth"], inplace=True)
filtered_df = filtered_df[["begin_day", "begin_time", "end_day", "end_time", "event_type", "begin_date_time", "cz_timezone", "end_date_time", "begin_lat", "begin_lon", "end_lat", "end_lon", "extreme"]]
filtered_df.dropna(inplace=True)
filtered_df.drop(columns=["begin_day","begin_time","end_day","end_time"], inplace=True)

import pandas as pd
from datetime import datetime
import pytz

df = filtered_df.copy()
# Mapping of timezone abbreviations to UTC offsets
timezone_offsets = {
    'CST-6': -6,
    'EST-5': -5,
    'MST-7': -7,
    'PST-8': -8,
    'AST-4': -4,
    'HST-10': -10,
    'AKST-9': -9,
    'SST-11': -11,
    'GST10': 10
}

# Function to convert local time to UTC
def convert_to_utc(local_time_str, timezone):
    local_time = datetime.strptime(local_time_str, '%d-%b-%y %H:%M:%S')
    offset = timezone_offsets.get(timezone, 0)
    local_time = local_time.replace(tzinfo=pytz.FixedOffset(offset * 60))
    utc_time = local_time.astimezone(pytz.utc)
    return utc_time

# Apply the function to the begin_date_time and end_date_time columns
df['begin_date_time_utc'] = df.apply(lambda row: convert_to_utc(row['begin_date_time'], row['cz_timezone']), axis=1)
df['end_date_time_utc'] = df.apply(lambda row: convert_to_utc(row['end_date_time'], row['cz_timezone']), axis=1)

# Split the datetime objects into separate date and time columns
df['begin_date_utc'] = df['begin_date_time_utc'].dt.date
df['begin_time_utc'] = df['begin_date_time_utc'].dt.time
df['end_date_utc'] = df['end_date_time_utc'].dt.date
df['end_time_utc'] = df['end_date_time_utc'].dt.time

# Drop the intermediate UTC datetime columns
df.drop(columns=['begin_date_time_utc', 'end_date_time_utc'], inplace=True)


/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_15249/2486818049.py:2: DtypeWarning: Columns (16,25,26,28,29,34,35,37,39,40,42,43,48,49,55,56,68,69) have mixed types. Specify dtype option on import or set low_memory=False.
  final_extreme_event_dataset = pd.read_csv("/Users/etloaner/Documents/ASU/Capstone project/final_storm_data_with_extreme_flag.csv")
/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_15249/2486818049.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.drop(columns=["episode_id_x","event_id","state_fips","cz_type","cz_fips","cz_name","wfo","source","tor_other_wfo","tor_other_cz_state","tor_other_cz_fips","tor_other_cz_name","episode_narrative","event_narrative","data_source","episode_id_y","azimuth"], inplace=True)


In [ ]:
df.to_csv("final_storm_data_with_utc.csv", index=False)

In [ ]:
df.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc
325600,Thunderstorm Wind,06-APR-17 15:09:00,EST-5,06-APR-17 15:09:00,39.6600,-75.0800,39.6600,-75.0800,1,2017-04-06,20:09:00,2017-04-06,20:09:00
325601,Tornado,06-APR-17 09:30:00,EST-5,06-APR-17 09:40:00,26.5010,-81.9980,26.5339,-81.8836,1,2017-04-06,14:30:00,2017-04-06,14:40:00
325602,Tornado,06-APR-17 09:30:00,EST-5,06-APR-17 09:40:00,26.5010,-81.9980,26.5339,-81.8836,1,2017-04-06,14:30:00,2017-04-06,14:40:00
325603,Thunderstorm Wind,05-APR-17 17:49:00,EST-5,05-APR-17 17:53:00,39.8500,-83.9900,39.8500,-83.9900,1,2017-04-05,22:49:00,2017-04-05,22:53:00
325604,Flood,16-APR-17 17:59:00,EST-5,16-APR-17 19:00:00,39.1065,-84.2875,39.1061,-84.2874,0,2017-04-16,22:59:00,2017-04-17,00:00:00


In [ ]:
import pandas as pd
df = pd.read_csv("final_storm_data_with_utc.csv")

# Convert the time columns to datetime objects
df['begin_date_time'] = pd.to_datetime(df['begin_date_time'], format='%d-%b-%y %H:%M:%S')
df['end_date_time'] = pd.to_datetime(df['end_date_time'], format='%d-%b-%y %H:%M:%S')

# Filter the DataFrame for events that lasted for some time (i.e., end time greater than start time)
filtered_df = df[df['end_date_time'] > df['begin_date_time']]

# Count the rows where the start and end times are exactly the same (i.e., instantaneous events)
same_time_count = df[df['begin_date_time'] == df['end_date_time']].shape[0]

# Display results
print("Filtered events (lasting more than 0 seconds):")
print(filtered_df)
print("\nNumber of rows with identical start and end times on the same day:", same_time_count)


Filtered events (lasting more than 0 seconds):
               event_type     begin_date_time cz_timezone       end_date_time  \
1                 Tornado 2017-04-06 09:30:00       EST-5 2017-04-06 09:40:00   
2                 Tornado 2017-04-06 09:30:00       EST-5 2017-04-06 09:40:00   
3       Thunderstorm Wind 2017-04-05 17:49:00       EST-5 2017-04-05 17:53:00   
4                   Flood 2017-04-16 17:59:00       EST-5 2017-04-16 19:00:00   
5                   Flood 2017-04-16 17:59:00       EST-5 2017-04-16 19:00:00   
...                   ...                 ...         ...                 ...   
577100               Hail 2012-04-03 13:21:00       CST-6 2012-04-03 13:25:00   
577105            Tornado 2012-04-03 13:08:00       CST-6 2012-04-03 13:13:00   
577106            Tornado 2012-04-03 13:08:00       CST-6 2012-04-03 13:13:00   
577107            Tornado 2012-04-03 14:24:00       CST-6 2012-04-03 14:25:00   
577108            Tornado 2012-04-03 14:24:00       CST-6 2012

In [ ]:
filtered_df = df[df['end_date_time'] > df['begin_date_time']]
same_time_count = df[df['begin_date_time'] == df['end_date_time']].shape[0]
print(filtered_df.shape[0])

370419


In [ ]:
import pandas as pd

# Convert the columns to datetime objects
df['begin_date_time'] = pd.to_datetime(df['begin_date_time'], format='%d-%b-%y %H:%M:%S')
df['end_date_time'] = pd.to_datetime(df['end_date_time'], format='%d-%b-%y %H:%M:%S')

# Define a 20 minutes threshold
time_threshold = pd.Timedelta(minutes=120)

# Filter the DataFrame for events with a duration longer than the threshold
filtered_df = df[(df['end_date_time'] - df['begin_date_time']) > time_threshold]

# Count the rows where the start and end times are exactly the same (instantaneous events)
same_time_count = df[df['begin_date_time'] == df['end_date_time']].shape[0]

# Display the filtered DataFrame and the count of instantaneous events
print("Filtered events (lasting more than 20 minutes):")
print(filtered_df)
print("\nNumber of rows with identical start and end times on the same day:", same_time_count)


Filtered events (lasting more than 20 minutes):
         event_type     begin_date_time cz_timezone       end_date_time  \
59      Flash Flood 2017-06-21 09:15:00       CST-6 2017-06-21 11:45:00   
60      Flash Flood 2017-06-21 09:15:00       CST-6 2017-06-21 11:45:00   
61      Flash Flood 2017-06-21 09:15:00       CST-6 2017-06-21 11:45:00   
62      Flash Flood 2017-06-21 09:15:00       CST-6 2017-06-21 11:45:00   
63            Flood 2017-06-22 05:00:00       CST-6 2017-06-23 05:00:00   
...             ...                 ...         ...                 ...   
576737        Flood 2012-04-27 18:00:00       MST-7 2012-04-28 12:00:00   
576738        Flood 2012-04-27 18:00:00       MST-7 2012-04-28 12:00:00   
576739        Flood 2012-04-27 18:00:00       MST-7 2012-04-28 12:00:00   
576740        Flood 2012-04-27 18:00:00       MST-7 2012-04-28 12:00:00   
576741        Flood 2012-04-27 18:00:00       MST-7 2012-04-28 12:00:00   

        begin_lat  begin_lon  end_lat   end_lon  ex

In [ ]:
filtered_df.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc
59,Flash Flood,2017-06-21 09:15:00,CST-6,2017-06-21 11:45:00,30.4546,-85.6994,30.4670,-85.6997,0,2017-06-21,15:15:00,2017-06-21,17:45:00
60,Flash Flood,2017-06-21 09:15:00,CST-6,2017-06-21 11:45:00,30.4546,-85.6994,30.4670,-85.6997,0,2017-06-21,15:15:00,2017-06-21,17:45:00
61,Flash Flood,2017-06-21 09:15:00,CST-6,2017-06-21 11:45:00,30.4546,-85.6994,30.4670,-85.6997,0,2017-06-21,15:15:00,2017-06-21,17:45:00
62,Flash Flood,2017-06-21 09:15:00,CST-6,2017-06-21 11:45:00,30.4546,-85.6994,30.4670,-85.6997,0,2017-06-21,15:15:00,2017-06-21,17:45:00
63,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.2746,-86.0081,30.2747,-86.0083,0,2017-06-22,11:00:00,2017-06-23,11:00:00


In [ ]:
filtered_df.drop_duplicates(inplace=True)

/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_16597/1486129318.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df.drop_duplicates(inplace=True)


In [ ]:
# Define grid size based on satellite resolution
import pandas as pd
import numpy as np
GRID_SIZE = 0.02  # approximately 2km in decimal degrees

# Round coordinates to match satellite data resolution
filtered_df['begin_lat'] = np.round(filtered_df['begin_lat'] / GRID_SIZE) * GRID_SIZE
filtered_df['begin_lon'] = np.round(filtered_df['begin_lon'] / GRID_SIZE) * GRID_SIZE
filtered_df['end_lat'] = np.round(filtered_df['end_lat'] / GRID_SIZE) * GRID_SIZE
filtered_df['end_lon'] = np.round(filtered_df['end_lon'] / GRID_SIZE) * GRID_SIZE

# Remove duplicates based on gridded coordinates
filtered_df = filtered_df.drop_duplicates(subset=[
    'event_type',
    'begin_date_time',
    'begin_lat',
    'begin_lon',
    'end_lat',
    'end_lon'
])

/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_16597/1936306879.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['begin_lat'] = np.round(filtered_df['begin_lat'] / GRID_SIZE) * GRID_SIZE
/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_16597/1936306879.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['begin_lon'] = np.round(filtered_df['begin_lon'] / GRID_SIZE) * GRID_SIZE
/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_16597/1936306879.py:9: Se

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
from math import radians, cos, sin, asin, sqrt

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance between two points in kilometers"""
    R = 6371  # Earth's radius in kilometers
    
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# Create coordinate array
coords = filtered_df[['begin_lat', 'begin_lon']].values

# DBSCAN with eps=0.018 (approximately 2km)
# eps is in degrees, 0.018 degrees ≈ 2km at the equator
clustering = DBSCAN(eps=0.018, min_samples=1).fit(coords)

# Add cluster labels to DataFrame
filtered_df['cluster'] = clustering.labels_

# Keep one event per cluster
filtered_df = filtered_df.groupby(['cluster', 'event_type', 'begin_date_time']).first().reset_index()
filtered_df.drop('cluster', axis=1, inplace=True)

In [ ]:
filtered_df.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc
0,Flash Flood,2017-06-21 09:15:00,CST-6,2017-06-21 11:45:00,30.46,-85.70,30.46,-85.70,0,2017-06-21,15:15:00,2017-06-21,17:45:00
1,Flood,2014-04-30 10:00:00,CST-6,2014-04-30 23:59:00,30.28,-86.00,30.12,-85.72,1,2014-04-30,16:00:00,2014-05-01,05:59:00
2,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.28,-86.00,30.28,-86.00,0,2017-06-22,11:00:00,2017-06-23,11:00:00
3,Flood,2014-04-18 13:20:00,CST-6,2014-04-19 00:00:00,30.98,-86.16,30.98,-86.16,0,2014-04-18,19:20:00,2014-04-19,06:00:00
4,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.98,-86.16,30.98,-86.16,0,2017-06-22,11:00:00,2017-06-23,11:00:00


In [ ]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41752 entries, 0 to 41751
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   event_type       41752 non-null  object        
 1   begin_date_time  41752 non-null  datetime64[ns]
 2   cz_timezone      41752 non-null  object        
 3   end_date_time    41752 non-null  datetime64[ns]
 4   begin_lat        41752 non-null  float64       
 5   begin_lon        41752 non-null  float64       
 6   end_lat          41752 non-null  float64       
 7   end_lon          41752 non-null  float64       
 8   extreme          41752 non-null  int64         
 9   begin_date_utc   41752 non-null  object        
 10  begin_time_utc   41752 non-null  object        
 11  end_date_utc     41752 non-null  object        
 12  end_time_utc     41752 non-null  object        
dtypes: datetime64[ns](2), float64(4), int64(1), object(6)
memory usage: 4.1+ MB


In [ ]:
from sklearn.cluster import KMeans
import folium
import numpy as np
from folium.plugins import MarkerCluster

# Extract coordinates for clustering
coordinates = filtered_df[['begin_lat', 'begin_lon']].values

# Perform KMeans clustering
kmeans = KMeans(n_clusters=50, random_state=42)
filtered_df['cluster'] = kmeans.fit_predict(coordinates)

# Get cluster centers
cluster_centers = kmeans.cluster_centers_

# Create a base map centered on the mean coordinates
center_lat = filtered_df['begin_lat'].mean()
center_lon = filtered_df['begin_lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=4)

# Add a MarkerCluster layer for all points
marker_cluster = MarkerCluster().add_to(m)

# Plot all points with cluster colors
colors = [f'#{hash(str(c)) % 0xFFFFFF:06x}' for c in range(50)]  # Generate unique colors

# Add individual points with cluster colors
for idx, row in filtered_df.iterrows():
    folium.CircleMarker(
        location=[row['begin_lat'], row['begin_lon']],
        radius=3,
        color=colors[row['cluster']],
        fill=True,
        popup=f"Cluster: {row['cluster']}<br>Event: {row['event_type']}"
    ).add_to(marker_cluster)

# Add cluster centers with larger markers
for i, center in enumerate(cluster_centers):
    folium.CircleMarker(
        location=[center[0], center[1]],
        radius=8,
        color='red',
        fill=True,
        popup=f'Cluster Center {i}'
    ).add_to(m)

# Save the map
m.save('weather_clusters.html')

# Print cluster statistics
print("\nCluster Statistics:")
cluster_stats = filtered_df.groupby('cluster').agg({
    'event_type': 'count',
    'begin_lat': ['mean', 'std'],
    'begin_lon': ['mean', 'std']
}).round(4)
print(cluster_stats)


Cluster Statistics:
        event_type begin_lat         begin_lon        
             count      mean     std      mean     std
cluster                                               
0              988   41.7701  1.0801  -87.3871  1.3015
1              250   43.6450  1.6516 -112.7131  1.5462
2              443   21.7837  0.3186 -158.6478  0.7736
3              664   43.4861  1.2956  -70.9841  1.0771
4             1266   37.1242  0.6134  -77.1953  0.8997
5             1652   37.5827  0.8988  -93.4279  0.9294
6              690   18.2877  0.1503  -66.2181  0.5356
7              500   41.6959  2.0522 -103.7326  1.1419
8               85   63.2492  2.5448 -148.0699  2.4589
9             1814   43.3035  0.8068  -96.6297  0.8973
10               8   13.4275  0.1136  144.7600  0.0920
11             842   31.9251  1.0577  -90.1526  1.0964
12            1194   34.6510  0.8016  -85.8605  1.3707
13             585   29.1862  1.4011  -98.2454  0.9655
14             578   34.5146  1.2244 -116.98

In [ ]:
filtered_df["extreme"].value_counts()

extreme
0    37697
1     4055
Name: count, dtype: int64

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def create_final_dataset(filtered_df, api_weather_data):
    """
    Merge extreme events data with weather API data
    """
    # Convert datetime columns to consistent format
    filtered_df['date'] = pd.to_datetime(filtered_df['begin_date_time'])
    api_weather_data['date'] = pd.to_datetime(api_weather_data['date'])
    
    # Round coordinates to 4 decimal places for matching
    filtered_df['lat_rounded'] = filtered_df['begin_lat'].round(4)
    filtered_df['lon_rounded'] = filtered_df['begin_lon'].round(4)
    api_weather_data['lat_rounded'] = api_weather_data['latitude'].round(4)
    api_weather_data['lon_rounded'] = api_weather_data['longitude'].round(4)
    
    # Merge datasets based on date and location
    merged_df = pd.merge(
        filtered_df,
        api_weather_data,
        how='left',
        left_on=['date', 'lat_rounded', 'lon_rounded'],
        right_on=['date', 'lat_rounded', 'lon_rounded']
    )
    
    # Clean up merged dataset
    # Drop duplicate columns and temporary columns
    columns_to_drop = ['lat_rounded', 'lon_rounded', 'city', 'latitude', 'longitude']
    merged_df = merged_df.drop(columns=columns_to_drop)
    
    # Check for missing values
    missing_data = merged_df.isnull().sum()
    print("\nMissing values in merged dataset:")
    print(missing_data[missing_data > 0])
    
    # Save final dataset
    merged_df.to_csv('final_weather_events_dataset.csv', index=False)
    print("\nFinal dataset shape:", merged_df.shape)
    
    return merged_df

# Load the API weather data
api_weather_data = pd.read_csv('us_weather_data.csv')

# Create final dataset
final_df = create_final_dataset(filtered_df, api_weather_data)

# Verify the merge
print("\nSample of final dataset:")
print(final_df.head())

# Check class distribution in final dataset
print("\nExtreme event distribution in final dataset:")
print(final_df['extreme'].value_counts(normalize=True))

In [ ]:
filtered_df.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster
0,Flash Flood,2017-06-21 09:15:00,CST-6,2017-06-21 11:45:00,30.46,-85.70,30.46,-85.70,0,2017-06-21,15:15:00,2017-06-21,17:45:00,45
1,Flood,2014-04-30 10:00:00,CST-6,2014-04-30 23:59:00,30.28,-86.00,30.12,-85.72,1,2014-04-30,16:00:00,2014-05-01,05:59:00,45
2,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.28,-86.00,30.28,-86.00,0,2017-06-22,11:00:00,2017-06-23,11:00:00,45
3,Flood,2014-04-18 13:20:00,CST-6,2014-04-19 00:00:00,30.98,-86.16,30.98,-86.16,0,2014-04-18,19:20:00,2014-04-19,06:00:00,45
4,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.98,-86.16,30.98,-86.16,0,2017-06-22,11:00:00,2017-06-23,11:00:00,45


In [ ]:
def get_unique_locations(filtered_df):
    """Extract unique locations and dates from events dataframe"""
    locations = filtered_df.groupby(['begin_lat', 'begin_lon']).agg({
        'begin_date_time': ['min', 'max']
    }).reset_index()
    
    locations.columns = ['lat', 'lon', 'start_date', 'end_date']
    locations['start_date'] = pd.to_datetime(locations['start_date']).dt.strftime('%Y-%m-%d')
    locations['end_date'] = pd.to_datetime(locations['end_date']).dt.strftime('%Y-%m-%d')
    
    return locations.to_dict('records')

# Get unique locations from the filtered dataset
unique_locations = get_unique_locations(filtered_df)

In [ ]:
len(unique_locations)

29255

In [ ]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import time
from tqdm import tqdm
from datetime import datetime, timedelta

def get_unique_locations_with_72h(filtered_df):
    """
    Extract unique locations and date ranges (previous 72 hours) from events dataframe.
    """
    # Add a column for 72 hours prior to each event's start time
    filtered_df['start_date_72h'] = pd.to_datetime(filtered_df['begin_date_time']) - timedelta(hours=72)
    filtered_df['start_date_72h'] = filtered_df['start_date_72h'].dt.strftime('%Y-%m-%d')
    filtered_df['end_date'] = pd.to_datetime(filtered_df['begin_date_time']).dt.strftime('%Y-%m-%d')
    
    # Group by locations and date ranges to minimize API calls
    locations = filtered_df.groupby(['begin_lat', 'begin_lon', 'start_date_72h', 'end_date']).size().reset_index()
    locations.columns = ['lat', 'lon', 'start_date', 'end_date', 'count']
    
    return locations.to_dict('records')

def fetch_weather_data_optimized(filtered_df, batch_size=10):
    """
    Fetch weather data for all unique locations with batching.
    This function minimizes API calls by grouping requests by location and date range.
    """
    # Setup the Open-Meteo API client with caching and retries
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)
    
    # Get unique locations with 72-hour ranges
    locations = get_unique_locations_with_72h(filtered_df)
    print(f"Total unique location-date combinations: {len(locations)}")
    
    # Initialize list for all weather data
    all_weather_data = []
    
    # Process locations in batches
    for i in tqdm(range(0, len(locations), batch_size), desc="Fetching weather data"):
        batch = locations[i:i + batch_size]
        
        for loc in batch:
            params = {
                "latitude": loc["lat"],
                "longitude": loc["lon"],
                "start_date": loc["start_date"],
                "end_date": loc["end_date"],
                "hourly": [
                    "temperature_2m",
                    "relative_humidity_2m",
                    "dew_point_2m",
                    "apparent_temperature",
                    "precipitation",
                    "rain",
                    "snowfall",
                    "snow_depth",
                    "weather_code",
                    "pressure_msl",
                    "surface_pressure",
                    "cloud_cover",
                    "wind_speed_10m",
                    "wind_direction_10m",
                    "wind_gusts_10m",
                    "soil_temperature_0_to_7cm",
                    "soil_moisture_0_to_7cm"
                ]
            }
            
            try:
                responses = openmeteo.weather_api(
                    "https://archive-api.open-meteo.com/v1/archive",
                    params=params
                )
                response = responses[0]
                
                # Process hourly data
                hourly = response.Hourly()
                data = {
                    "date": pd.date_range(
                        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
                        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                        freq=pd.Timedelta(seconds=hourly.Interval()),
                        inclusive="left"
                    ),
                    "latitude": loc["lat"],
                    "longitude": loc["lon"]
                }
                
                # Add all weather variables
                for idx, var in enumerate(params["hourly"]):
                    data[var] = hourly.Variables(idx).ValuesAsNumpy()
                
                all_weather_data.append(pd.DataFrame(data))
                
            except Exception as e:
                print(f"\nError fetching data for location {loc['lat']}, {loc['lon']}: {str(e)}")
                continue
        
        # Rate limiting to avoid hitting API limits
        time.sleep(1)
    
    # Combine all data into a single DataFrame
    if all_weather_data:
        weather_df = pd.concat(all_weather_data, ignore_index=True)
        weather_df.to_csv('weather_data.csv', index=False)
        print(f"\nCollected weather data shape: {weather_df.shape}")
        return weather_df
    else:
        raise Exception("No weather data collected")

def create_final_dataset(events_df, weather_df):
    """
    Merge the events dataframe with the collected weather data.
    """
    # Convert timestamps to match for merging
    events_df['datetime'] = pd.to_datetime(events_df['begin_date_time'])
    
    # Merge on latitude, longitude, and datetime (nearest match within 1 hour)
    final_dataset = pd.merge_asof(
        events_df.sort_values('datetime'),
        weather_df.sort_values('date'),
        left_on='datetime',
        right_on='date',
        by=['begin_lat', 'begin_lon'],
        tolerance=pd.Timedelta('1H'),
        direction='nearest'
    )
    
    return final_dataset

# Main execution
def main():
    try:
        # Load your events dataframe (replace this with your actual dataframe)
        filtered_df = pd.read_csv("events.csv")  # Replace with your file path
        
        # Fetch weather data (optimized to minimize API calls)
        weather_data = fetch_weather_data_optimized(filtered_df)
        
        # Create final dataset by merging events with weather data
        final_df = create_final_dataset(filtered_df, weather_data)
        
        print("\nData collection and merging completed successfully!")
        
        # Save final dataset to a CSV file
        final_df.to_csv("final_dataset.csv", index=False)
        
        return final_df
    
    except Exception as e:
        print(f"Error in data collection process: {str(e)}")
        return None

# Run the process
if __name__ == "__main__":
    final_df = main()


In [ ]:

import pandas as pd
from datetime import datetime, timedelta
def get_unique_locations_with_72h(filtered_df):
    """
    Extract unique locations and date ranges (previous 72 hours) from events dataframe.
    """
    # Add a column for 72 hours prior to each event's start time
    filtered_df['start_date_72h'] = pd.to_datetime(filtered_df['begin_date_time']) - timedelta(hours=72)
    filtered_df['start_date_72h'] = filtered_df['start_date_72h'].dt.strftime('%Y-%m-%d')
    filtered_df['end_date'] = pd.to_datetime(filtered_df['begin_date_time']).dt.strftime('%Y-%m-%d')
    
    # Group by locations and date ranges to minimize API calls
    locations = filtered_df.groupby(['begin_lat', 'begin_lon', 'start_date_72h', 'end_date']).size().reset_index()
    locations.columns = ['lat', 'lon', 'start_date', 'end_date', 'count']
    
    return locations.to_dict('records')

get_unique_locations_with_72h(filtered_df)

[{'lat': -14.4,
  'lon': -170.70000000000002,
  'start_date': '2014-01-05',
  'end_date': '2014-01-08',
  'count': 1},
 {'lat': -14.36,
  'lon': -170.76,
  'start_date': '2017-12-07',
  'end_date': '2017-12-10',
  'count': 1},
 {'lat': -14.34,
  'lon': -170.84,
  'start_date': '2013-12-06',
  'end_date': '2013-12-09',
  'count': 1},
 {'lat': -14.34,
  'lon': -170.82,
  'start_date': '2017-10-15',
  'end_date': '2017-10-18',
  'count': 1},
 {'lat': -14.34,
  'lon': -170.82,
  'start_date': '2017-11-20',
  'end_date': '2017-11-23',
  'count': 1},
 {'lat': -14.34,
  'lon': -170.82,
  'start_date': '2018-01-02',
  'end_date': '2018-01-05',
  'count': 1},
 {'lat': -14.34,
  'lon': -170.82,
  'start_date': '2020-01-13',
  'end_date': '2020-01-16',
  'count': 1},
 {'lat': -14.34,
  'lon': -170.8,
  'start_date': '2017-08-16',
  'end_date': '2017-08-19',
  'count': 1},
 {'lat': -14.34,
  'lon': -170.78,
  'start_date': '2018-07-29',
  'end_date': '2018-08-01',
  'count': 1},
 {'lat': -14.34,
 

In [ ]:
len(get_unique_locations_with_72h(filtered_df))

40713

In [ ]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41752 entries, 0 to 41751
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   event_type       41752 non-null  object        
 1   begin_date_time  41752 non-null  datetime64[ns]
 2   cz_timezone      41752 non-null  object        
 3   end_date_time    41752 non-null  datetime64[ns]
 4   begin_lat        41752 non-null  float64       
 5   begin_lon        41752 non-null  float64       
 6   end_lat          41752 non-null  float64       
 7   end_lon          41752 non-null  float64       
 8   extreme          41752 non-null  int64         
 9   begin_date_utc   41752 non-null  object        
 10  begin_time_utc   41752 non-null  object        
 11  end_date_utc     41752 non-null  object        
 12  end_time_utc     41752 non-null  object        
 13  cluster          41752 non-null  int32         
 14  start_date_72h   41752 non-null  objec

In [ ]:
filtered_df.to_csv("Final_prediction_dataset.csv", index=False)

In [ ]:
filtered_df.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date
0,Flash Flood,2017-06-21 09:15:00,CST-6,2017-06-21 11:45:00,30.46,-85.70,30.46,-85.70,0,2017-06-21,15:15:00,2017-06-21,17:45:00,45,2017-06-18,2017-06-21
1,Flood,2014-04-30 10:00:00,CST-6,2014-04-30 23:59:00,30.28,-86.00,30.12,-85.72,1,2014-04-30,16:00:00,2014-05-01,05:59:00,45,2014-04-27,2014-04-30
2,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.28,-86.00,30.28,-86.00,0,2017-06-22,11:00:00,2017-06-23,11:00:00,45,2017-06-19,2017-06-22
3,Flood,2014-04-18 13:20:00,CST-6,2014-04-19 00:00:00,30.98,-86.16,30.98,-86.16,0,2014-04-18,19:20:00,2014-04-19,06:00:00,45,2014-04-15,2014-04-18
4,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.98,-86.16,30.98,-86.16,0,2017-06-22,11:00:00,2017-06-23,11:00:00,45,2017-06-19,2017-06-22


In [17]:
extreme_df = pd.read_csv("Final_prediction_dataset.csv")
extreme_df = extreme_df[:5]
extreme_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   event_type       5 non-null      object 
 1   begin_date_time  5 non-null      object 
 2   cz_timezone      5 non-null      object 
 3   end_date_time    5 non-null      object 
 4   begin_lat        5 non-null      float64
 5   begin_lon        5 non-null      float64
 6   end_lat          5 non-null      float64
 7   end_lon          5 non-null      float64
 8   extreme          5 non-null      int64  
 9   begin_date_utc   5 non-null      object 
 10  begin_time_utc   5 non-null      object 
 11  end_date_utc     5 non-null      object 
 12  end_time_utc     5 non-null      object 
 13  cluster          5 non-null      int64  
 14  start_date_72h   5 non-null      object 
 15  end_date         5 non-null      object 
dtypes: float64(4), int64(2), object(10)
memory usage: 768.0+ bytes


In [ ]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import time
import pickle
from tqdm import tqdm

# ----------------------------
# Setup API client with caching and retry logic
# ----------------------------
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)
url = "https://archive-api.open-meteo.com/v1/archive"

# ----------------------------
# Load the extreme event data
# ----------------------------
extreme_df = pd.read_csv("Final_prediction_dataset.csv")
# Create a UTC datetime column using the provided UTC date and time fields.
extreme_df['event_datetime'] = pd.to_datetime(
    extreme_df['begin_date_utc'] + " " + extreme_df['begin_time_utc'], utc=True
)

# ----------------------------
# Define the hourly weather variables to request.
# ----------------------------
hourly_vars = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "precipitation", "rain", "snowfall", "snow_depth", "weather_code", "pressure_msl",
    "surface_pressure", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "et0_fao_evapotranspiration", "vapour_pressure_deficit", "wind_speed_10m", "wind_speed_100m",
    "wind_direction_10m", "wind_direction_100m", "wind_gusts_10m", "soil_temperature_0_to_7cm",
    "soil_temperature_7_to_28cm", "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm", "soil_moisture_28_to_100cm",
    "soil_moisture_100_to_255cm"
]

# ----------------------------
# Process each extreme event (row) individually
# ----------------------------
prev_weather_list = []  # To store each event's 72-hour weather history.
failed_rows = []        # To log any failed API calls.

for idx, row in tqdm(extreme_df.iterrows(), total=len(extreme_df), desc="Processing events"):
    lat = row['begin_lat']
    lon = row['begin_lon']
    event_dt = row['event_datetime']
    
    # Calculate the start of the 72-hour window
    start_dt = event_dt - pd.Timedelta(hours=72)
    
    # Format the dates as required by the API (YYYY-MM-DD)
    start_date_str = start_dt.strftime('%Y-%m-%d')
    end_date_str = event_dt.strftime('%Y-%m-%d')
    
    # Set API parameters for this event
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date_str,
        "end_date": end_date_str,
        "hourly": hourly_vars
    }
    
    try:
        # Call the API (each call returns a list, so we take the first response)
        responses = openmeteo.weather_api(url, params=params)
        if not responses:
            raise ValueError("Empty response received")
        response = responses[0]
        hourly = response.Hourly()
        
        # Create a date_range from the hourly data timestamps (all in UTC)
        date_range = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        )
        
        # Build a DataFrame from the hourly data
        weather_data = {"date": date_range}
        for i, var in enumerate(hourly_vars):
            weather_data[var] = hourly.Variables(i).ValuesAsNumpy()
        weather_df = pd.DataFrame(weather_data)
        
        # Filter the DataFrame to only include records within the previous 72 hours
        mask = (weather_df['date'] >= start_dt) & (weather_df['date'] < event_dt)
        history_df = weather_df.loc[mask]
        history_records = history_df.to_dict(orient="records")
        prev_weather_list.append(history_records)
        
    except Exception as e:
        # Log error details and store None for this event
        failed_rows.append({
            "index": idx,
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date_str,
            "end_date": end_date_str,
            "error": str(e)
        })
        prev_weather_list.append(None)
        # Save partial data immediately in case of error
        with open("prev_weather_partial.pkl", "wb") as f:
            pickle.dump(prev_weather_list, f)
        print(f"Error at row {idx}: {e}. Partial data saved to 'prev_weather_partial.pkl'.")
    
    # Sleep 0.1 seconds to not exceed 600 calls per minute
    time.sleep(2)

# ----------------------------
# Save the collected weather history back into the extreme event DataFrame
# ----------------------------
extreme_df['prev_72h_weather'] = prev_weather_list

# Save the final DataFrame (using pickle to preserve the nested structure)
extreme_df.to_pickle("extreme_events_with_prev_weather.pkl")
print("Final data saved to 'extreme_events_with_prev_weather.pkl'.")

# Optionally, save the failed rows details to CSV for further review.
if failed_rows:
    pd.DataFrame(failed_rows).to_csv("failed_rows.csv", index=False)
    print("Some API calls failed. Details saved to 'failed_rows.csv'.")
else:
    print("All API calls succeeded.")


Processing events:   8%|▊         | 3134/41752 [2:06:34<31:10:59,  2.91s/it] 

In [28]:
extreme_df['prev_72h_weather'][0][-2]

{'date': Timestamp('2017-06-21 14:00:00+0000', tz='UTC'),
 'temperature_2m': 24.886999130249023,
 'relative_humidity_2m': 94.46976470947266,
 'dew_point_2m': 23.937000274658203,
 'apparent_temperature': 28.34389877319336,
 'precipitation': 3.5,
 'rain': 3.5,
 'snowfall': 0.0,
 'snow_depth': 0.0,
 'weather_code': 63.0,
 'pressure_msl': 1015.0999755859375,
 'surface_pressure': 1010.8057861328125,
 'cloud_cover': 100.0,
 'cloud_cover_low': 5.0,
 'cloud_cover_mid': 100.0,
 'cloud_cover_high': 100.0,
 'et0_fao_evapotranspiration': 0.04889203980565071,
 'vapour_pressure_deficit': 0.17386507987976074,
 'wind_speed_10m': 17.87355613708496,
 'wind_speed_100m': 29.96445655822754,
 'wind_direction_10m': 124.33027648925781,
 'wind_direction_100m': 125.21768188476562,
 'wind_gusts_10m': 42.47999954223633,
 'soil_temperature_0_to_7cm': 23.98699951171875,
 'soil_temperature_7_to_28cm': 23.687000274658203,
 'soil_temperature_28_to_100cm': 24.136999130249023,
 'soil_temperature_100_to_255cm': 22.086999

In [26]:
extreme_df

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date,event_datetime,prev_72h_weather
0,Flash Flood,2017-06-21 09:15:00,CST-6,2017-06-21 11:45:00,30.46,-85.70,30.46,-85.70,0,2017-06-21,15:15:00,2017-06-21,17:45:00,45,2017-06-18,2017-06-21,2017-06-21 15:15:00+00:00,"[{'date': 2017-06-18 16:00:00+00:00, 'temperat..."
1,Flood,2014-04-30 10:00:00,CST-6,2014-04-30 23:59:00,30.28,-86.00,30.12,-85.72,1,2014-04-30,16:00:00,2014-05-01,05:59:00,45,2014-04-27,2014-04-30,2014-04-30 16:00:00+00:00,"[{'date': 2014-04-27 16:00:00+00:00, 'temperat..."
2,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.28,-86.00,30.28,-86.00,0,2017-06-22,11:00:00,2017-06-23,11:00:00,45,2017-06-19,2017-06-22,2017-06-22 11:00:00+00:00,"[{'date': 2017-06-19 11:00:00+00:00, 'temperat..."
3,Flood,2014-04-18 13:20:00,CST-6,2014-04-19 00:00:00,30.98,-86.16,30.98,-86.16,0,2014-04-18,19:20:00,2014-04-19,06:00:00,45,2014-04-15,2014-04-18,2014-04-18 19:20:00+00:00,"[{'date': 2014-04-15 20:00:00+00:00, 'temperat..."
4,Flood,2017-06-22 05:00:00,CST-6,2017-06-23 05:00:00,30.98,-86.16,30.98,-86.16,0,2017-06-22,11:00:00,2017-06-23,11:00:00,45,2017-06-19,2017-06-22,2017-06-22 11:00:00+00:00,"[{'date': 2017-06-19 11:00:00+00:00, 'temperat..."


In [ ]:
import os
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import time
import pickle
from tqdm import tqdm
import logging

# Setup logging to output errors and progress.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Setup API client with caching and retry logic.
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)
url = "https://archive-api.open-meteo.com/v1/archive"

# Load the extreme event data.
extreme_df = pd.read_csv("Final_prediction_dataset.csv")
extreme_df = extreme_df.iloc[:10001,:]
extreme_df['event_datetime'] = pd.to_datetime(
    extreme_df['begin_date_utc'] + " " + extreme_df['begin_time_utc'], utc=True
)

# Define the hourly weather variables.
hourly_vars = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m", "apparent_temperature",
    "precipitation", "rain", "snowfall", "snow_depth", "weather_code", "pressure_msl",
    "surface_pressure", "cloud_cover", "cloud_cover_low", "cloud_cover_mid", "cloud_cover_high",
    "et0_fao_evapotranspiration", "vapour_pressure_deficit", "wind_speed_10m", "wind_speed_100m",
    "wind_direction_10m", "wind_direction_100m", "wind_gusts_10m", "soil_temperature_0_to_7cm",
    "soil_temperature_7_to_28cm", "soil_temperature_28_to_100cm", "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm", "soil_moisture_7_to_28cm", "soil_moisture_28_to_100cm",
    "soil_moisture_100_to_255cm"
]

# Attempt to load previous progress if available
if os.path.exists("prev_weather_partial.pkl"):
    with open("prev_weather_partial.pkl", "rb") as f:
        prev_weather_list = pickle.load(f)
    logging.info(f"Resuming from checkpoint. Processed events: {len(prev_weather_list)}.")
else:
    prev_weather_list = []

failed_rows = []

def save_progress(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f)
    logging.info(f"Progress saved to '{filename}'.")

# Determine starting index based on previously processed rows.
start_index = len(prev_weather_list)

# Resume processing from the start_index.
for idx, row in tqdm(extreme_df.iloc[start_index:].iterrows(), total=len(extreme_df) - start_index, desc="Processing events"):
    lat = row['begin_lat']
    lon = row['begin_lon']
    event_dt = row['event_datetime']
    
    start_dt = event_dt - pd.Timedelta(hours=72)
    start_date_str = start_dt.strftime('%Y-%m-%d')
    end_date_str = event_dt.strftime('%Y-%m-%d')
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date_str,
        "end_date": end_date_str,
        "hourly": hourly_vars
    }
    
    try:
        responses = openmeteo.weather_api(url, params=params)
        if not responses:
            raise ValueError("Empty response received")
        response = responses[0]
        print(response)
        break
        hourly = response.Hourly()
        
        # Create a date_range from the hourly data timestamps.
        date_range = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        )
        
        # Build a DataFrame from the hourly data.
        weather_data = {"date": date_range}
        for i, var in enumerate(hourly_vars):
            weather_data[var] = hourly.Variables(i).ValuesAsNumpy()
        weather_df = pd.DataFrame(weather_data)
        
        # Filter for records within the previous 72 hours.
        mask = (weather_df['date'] >= start_dt) & (weather_df['date'] < event_dt)
        history_df = weather_df.loc[mask]
        history_records = history_df.to_dict(orient="records")
        prev_weather_list.append(history_records)
        
    except Exception as e:
        failed_rows.append({
            "index": idx + start_index,  # adjust index to account for skipped rows
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date_str,
            "end_date": end_date_str,
            "error": str(e)
        })
        prev_weather_list.append(None)
        logging.error(f"Error at row {idx + start_index}: {e}. Saving partial data.")
        save_progress(prev_weather_list, "prev_weather_partial.pkl")
        time.sleep(5)
        continue
    
    # Sleep to control API call frequency.

    time.sleep(2)
    # Optionally save progress every N events.
    if (idx + start_index) % 50 == 0:
        save_progress(prev_weather_list, "prev_weather_partial.pkl")

extreme_df['prev_72h_weather'] = prev_weather_list
extreme_df.to_pickle("extreme_events_with_prev_weather.pkl")
logging.info("Final data saved to 'extreme_events_with_prev_weather.pkl'.")

if failed_rows:
    pd.DataFrame(failed_rows).to_csv("failed_rows.csv", index=False)
    logging.info("Some API calls failed. Details saved to 'failed_rows.csv'.")
else:
    logging.info("All API calls succeeded.")


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x10354b430>>
Traceback (most recent call last):
  File "/Users/etloaner/Library/Python/3.10/lib/python/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
2025-03-06 11:40:09,816 - INFO - Resuming from checkpoint. Processed events: 7014.
Processing events:   0%|          | 0/2987 [00:01<?, ?it/s]

ValueError: Length of values (7014) does not match length of index (10001)

Google Query: <think>
Okay, so the user wants to know how the weather is in Tempe today. They're asking me what query they should use to get that information from Google. Let's break it down.

First, I need to think about the key components of a typical weather query. Usually, people search for something like "weather [location] today" or similar phrasing. In this case, the location is Tempe. 

I remember that in many places, the capital letter is used for cities and places, so Tempe would be "Tempe," but it's actually spelled with a lowercase 't' sometimes. However, Google generally handles both correctly, so maybe I should include just "tempe" regardless of case.

The user specifically mentioned "how is the westher in tempe todat." Wait, that might be a typo for "weather." It's possible they meant "weather" but spelled it incorrectly as "westher." Also, "toadit"
Google Results: ['https://www.reddit.com/r/googlehome/comments/lwu5x2/asking_any_of_my_google_minis_or_home_units_whats/', 

In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [6]:
import pickle
with open("prev_weather_partial.pkl", "rb") as f:
    prev_weather_list = pickle.load(f)

In [9]:
len(prev_weather_list)

7014

ModuleNotFoundError: No module named 'owslib'

In [11]:
import satlaspretrain_models
import torch
weights_manager = satlaspretrain_models.Weights()

In [14]:
model = weights_manager.get_pretrained_model(model_identifier="Sentinel2_SwinB_SI_RGB", fpn=True, device="mps")

# Expected input is a portion of a Sentinel-2 L1C TCI image.
# The 0-255 pixel values should be divided by 255 so they are 0-1.
# tensor = tci_image[None, :, :, :] / 255
tensor = torch.zeros((1, 3, 512, 512), dtype=torch.float32)

# Since we only loaded the backbone, it outputs feature maps from the Swin-v2-Base backbone.
output = model(tensor)
print([feature_map.shape for feature_map in output])
# [torch.Size([1, 128, 128, 128]), torch.Size([1, 256, 64, 64]), torch.Size([1, 512, 32, 32]), torch.Size([1, 1024, 16, 16])]

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

## combining all data

In [43]:
import pickle


extreme_events_10001_15000 = pickle.load(open("extreme_events_10001_15000.pkl", "rb"))
extreme_events_15001_19448 = pickle.load(open("extreme_events_15001_19448.pkl", "rb"))
extreme_events_19449_20000 = pickle.load(open("extreme_events_19449_20000.pkl", "rb"))
extreme_events_20001_30000 = pickle.load(open("extreme_events_with_prev_weather (1)_20001_30000.pkl", "rb"))
extreme_events_30001_40000 = pickle.load(open("extreme_events_with_prev_weather (30000 - 40000).pkl", "rb"))

/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_55062/145692632.py:7: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  extreme_events_20001_30000 = pickle.load(open("extreme_events_with_prev_weather (1)_20001_30000.pkl", "rb"))


In [31]:
%pip install --upgrade pandas


[notice] A new release of pip available: 22.2.2 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [44]:
extreme_events_10001_15000.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date,event_datetime,prev_72h_weather
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,17:30:00,2012-10-30,03:30:00,4,2012-10-26,2012-10-29,2012-10-29 17:30:00+00:00,"[{'date': 2012-10-26 18:00:00+00:00, 'temperat..."
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,19:00:00,2014-05-16,22:00:00,4,2014-05-12,2014-05-15,2014-05-15 19:00:00+00:00,"[{'date': 2014-05-12 19:00:00+00:00, 'temperat..."
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,15:00:00,2016-10-09,16:00:00,4,2016-10-05,2016-10-08,2016-10-08 15:00:00+00:00,"[{'date': 2016-10-05 15:00:00+00:00, 'temperat..."
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,06:14:00,2014-09-22,09:00:00,19,2014-09-18,2014-09-21,2014-09-22 06:14:00+00:00,"[{'date': 2014-09-19 07:00:00+00:00, 'temperat..."
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,23:00:00,2014-09-28,06:00:00,48,2014-09-24,2014-09-27,2014-09-27 23:00:00+00:00,"[{'date': 2014-09-24 23:00:00+00:00, 'temperat..."


In [3]:
import pandas as pd
import pickle

with open("extreme_events_with_prev_weather (1).pkl", "rb") as f:
    bhavya = pd.read_pickle(f)

In [4]:
bhavya

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date,event_datetime,prev_72h_weather
6899,Flood,2018-06-03 14:19:00,EST-5,2018-06-03 19:28:00,38.96,-77.04,38.98,-77.04,0,2018-06-03,19:19:00,2018-06-04,00:28:00,36,2018-05-31,2018-06-03,2018-06-03 19:19:00+00:00,"[{'date': 2016-07-24 12:00:00+00:00, 'temperat..."
6900,Flood,2018-07-21 19:19:00,EST-5,2018-07-22 06:43:00,38.96,-77.04,38.98,-77.04,0,2018-07-22,00:19:00,2018-07-22,11:43:00,36,2018-07-18,2018-07-21,2018-07-22 00:19:00+00:00,"[{'date': 2016-07-19 01:00:00+00:00, 'temperat..."
6901,Flood,2018-11-24 18:44:00,EST-5,2018-11-25 01:16:00,38.96,-77.04,38.96,-77.04,0,2018-11-24,23:44:00,2018-11-25,06:16:00,36,2018-11-21,2018-11-24,2018-11-24 23:44:00+00:00,"[{'date': 2016-06-30 01:00:00+00:00, 'temperat..."
6902,Flood,2018-12-15 20:25:00,EST-5,2018-12-16 07:30:00,38.96,-77.04,38.96,-77.04,0,2018-12-16,01:25:00,2018-12-16,12:30:00,36,2018-12-12,2018-12-15,2018-12-16 01:25:00+00:00,"[{'date': 2016-07-04 12:00:00+00:00, 'temperat..."
6903,Flood,2019-03-21 18:15:00,EST-5,2019-03-22 01:35:00,38.96,-77.04,38.96,-77.04,0,2019-03-21,23:15:00,2019-03-22,06:35:00,36,2019-03-18,2019-03-21,2019-03-21 23:15:00+00:00,"[{'date': 2016-07-27 05:00:00+00:00, 'temperat..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41747,Flood,2012-09-09 23:30:00,MST-7,2012-09-13 08:30:00,31.96,-112.20,31.98,-112.36,0,2012-09-10,06:30:00,2012-09-13,15:30:00,26,2012-09-06,2012-09-09,2012-09-10 06:30:00+00:00,"[{'date': 2012-09-07 07:00:00+00:00, 'temperat..."
41748,Flash Flood,2012-03-21 19:30:00,CST-6,2012-03-21 22:40:00,31.60,-89.04,31.76,-88.98,0,2012-03-22,01:30:00,2012-03-22,04:40:00,11,2012-03-18,2012-03-21,2012-03-22 01:30:00+00:00,"[{'date': 2012-03-19 02:00:00+00:00, 'temperat..."
41749,Flash Flood,2012-09-17 15:25:00,CST-6,2012-09-18 06:00:00,35.48,-86.44,35.48,-86.44,0,2012-09-17,21:25:00,2012-09-18,12:00:00,12,2012-09-14,2012-09-17,2012-09-17 21:25:00+00:00,"[{'date': 2012-09-14 22:00:00+00:00, 'temperat..."
41750,Flood,2012-04-26 20:00:00,MST-7,2012-04-28 01:00:00,43.92,-116.44,43.86,-116.56,0,2012-04-27,03:00:00,2012-04-28,08:00:00,42,2012-04-23,2012-04-26,2012-04-27 03:00:00+00:00,"[{'date': 2012-04-24 03:00:00+00:00, 'temperat..."


In [7]:
len(bhavya)

4853

In [5]:
import pandas as pd
extreme_df = pd.read_csv("Final_prediction_dataset.csv")

In [23]:
print(len(extreme_events_10001_15000))
print(len(extreme_events_15001_19448))
print(len(extreme_events_19449_20000))
print(len(extreme_events_20001_30000))
print(len(extreme_events_30001_40000))

5000
5000
552
9999
10000


In [6]:
extreme_events_30001_40000

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date,event_datetime,prev_72h_weather
0,Flash Flood,2013-06-19 07:25:00,CST-6,2013-06-19 10:34:00,34.94,-100.90,34.94,-100.90,0,2013-06-19,13:25:00,2013-06-19,16:34:00,37,2013-06-16,2013-06-19,2013-06-19 13:25:00+00:00,"[{'date': 2013-06-16 14:00:00+00:00, 'temperat..."
1,Flash Flood,2013-09-28 01:55:00,CST-6,2013-09-28 04:00:00,34.94,-100.90,34.94,-100.88,0,2013-09-28,07:55:00,2013-09-28,10:00:00,37,2013-09-25,2013-09-28,2013-09-28 07:55:00+00:00,"[{'date': 2013-09-25 08:00:00+00:00, 'temperat..."
2,Flood,2013-06-13 02:27:00,EST-5,2013-06-13 08:00:00,40.26,-81.88,40.24,-81.86,0,2013-06-13,07:27:00,2013-06-13,13:00:00,18,2013-06-10,2013-06-13,2013-06-13 07:27:00+00:00,"[{'date': 2013-06-10 08:00:00+00:00, 'temperat..."
3,Flood,2013-06-13 02:33:00,EST-5,2013-06-13 08:00:00,40.26,-81.72,40.26,-81.70,0,2013-06-13,07:33:00,2013-06-13,13:00:00,18,2013-06-10,2013-06-13,2013-06-13 07:33:00+00:00,"[{'date': 2013-06-10 08:00:00+00:00, 'temperat..."
4,Flash Flood,2013-07-02 18:21:00,EST-5,2013-07-02 20:30:00,43.02,-73.84,43.02,-73.84,0,2013-07-02,23:21:00,2013-07-03,01:30:00,21,2013-06-29,2013-07-02,2013-07-02 23:21:00+00:00,"[{'date': 2013-06-30 00:00:00+00:00, 'temperat..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Flood,2016-07-08 02:43:00,EST-5,2016-07-08 08:00:00,42.58,-82.88,42.56,-83.04,0,2016-07-08,07:43:00,2016-07-08,13:00:00,18,2016-07-05,2016-07-08,2016-07-08 07:43:00+00:00,"[{'date': 2016-07-05 08:00:00+00:00, 'temperat..."
9996,Flash Flood,2016-07-31 19:00:00,MST-7,2016-07-31 22:00:00,45.88,-103.70,45.74,-103.66,0,2016-08-01,02:00:00,2016-08-01,05:00:00,38,2016-07-28,2016-07-31,2016-08-01 02:00:00+00:00,"[{'date': 2016-07-29 02:00:00+00:00, 'temperat..."
9997,Heavy Rain,2016-07-31 16:30:00,MST-7,2016-07-31 20:30:00,45.80,-103.60,45.80,-103.60,0,2016-07-31,23:30:00,2016-08-01,03:30:00,38,2016-07-28,2016-07-31,2016-07-31 23:30:00+00:00,"[{'date': 2016-07-29 00:00:00+00:00, 'temperat..."
9998,Heavy Rain,2016-07-06 05:00:00,CST-6,2016-07-07 05:00:00,36.44,-86.80,36.44,-86.80,0,2016-07-06,11:00:00,2016-07-07,11:00:00,29,2016-07-03,2016-07-06,2016-07-06 11:00:00+00:00,"[{'date': 2016-07-03 11:00:00+00:00, 'temperat..."


In [7]:
final_ds = pd.concat([extreme_events_10001_15000, extreme_events_15001_19448, extreme_events_19449_20000, extreme_events_20001_30000, extreme_events_30001_40000, bhavya], ignore_index=True)

In [8]:
final_ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35404 entries, 0 to 35403
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   event_type        35404 non-null  object             
 1   begin_date_time   35404 non-null  object             
 2   cz_timezone       35404 non-null  object             
 3   end_date_time     35404 non-null  object             
 4   begin_lat         35404 non-null  float64            
 5   begin_lon         35404 non-null  float64            
 6   end_lat           35404 non-null  float64            
 7   end_lon           35404 non-null  float64            
 8   extreme           35404 non-null  int64              
 9   begin_date_utc    35404 non-null  object             
 10  begin_time_utc    35404 non-null  object             
 11  end_date_utc      35404 non-null  object             
 12  end_time_utc      35404 non-null  object             
 13  c

In [9]:
final_ds.dropna(inplace=True)

In [10]:
final_ds.info()

<class 'pandas.core.frame.DataFrame'>
Index: 35137 entries, 0 to 35403
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   event_type        35137 non-null  object             
 1   begin_date_time   35137 non-null  object             
 2   cz_timezone       35137 non-null  object             
 3   end_date_time     35137 non-null  object             
 4   begin_lat         35137 non-null  float64            
 5   begin_lon         35137 non-null  float64            
 6   end_lat           35137 non-null  float64            
 7   end_lon           35137 non-null  float64            
 8   extreme           35137 non-null  int64              
 9   begin_date_utc    35137 non-null  object             
 10  begin_time_utc    35137 non-null  object             
 11  end_date_utc      35137 non-null  object             
 12  end_time_utc      35137 non-null  object             
 13  cluste

In [11]:
final_ds.to_csv("final_ds.csv", index=False)

In [39]:
final_ds = pd.read_csv("final_ds.csv")

In [40]:
final_ds.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date,event_datetime,prev_72h_weather
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,17:30:00,2012-10-30,03:30:00,4,2012-10-26,2012-10-29,2012-10-29 17:30:00+00:00,[{'date': Timestamp('2012-10-26 18:00:00+0000'...
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,19:00:00,2014-05-16,22:00:00,4,2014-05-12,2014-05-15,2014-05-15 19:00:00+00:00,[{'date': Timestamp('2014-05-12 19:00:00+0000'...
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,15:00:00,2016-10-09,16:00:00,4,2016-10-05,2016-10-08,2016-10-08 15:00:00+00:00,[{'date': Timestamp('2016-10-05 15:00:00+0000'...
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,06:14:00,2014-09-22,09:00:00,19,2014-09-18,2014-09-21,2014-09-22 06:14:00+00:00,[{'date': Timestamp('2014-09-19 07:00:00+0000'...
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,23:00:00,2014-09-28,06:00:00,48,2014-09-24,2014-09-27,2014-09-27 23:00:00+00:00,[{'date': Timestamp('2014-09-24 23:00:00+0000'...


In [24]:
final_ds.head()

NameError: name 'final_ds' is not defined

In [15]:
final_ds.iloc[0, :]

event_type                                                      Flood
begin_date_time                                   2012-10-29 12:30:00
cz_timezone                                                     EST-5
end_date_time                                     2012-10-29 22:30:00
begin_lat                                                       37.84
begin_lon                                                      -75.48
end_lat                                                          37.7
end_lon                                                        -75.62
extreme                                                             0
begin_date_utc                                             2012-10-29
begin_time_utc                                               17:30:00
end_date_utc                                               2012-10-30
end_time_utc                                                 03:30:00
cluster                                                             4
start_date_72h      

In [5]:
import pandas as pd
from datetime import datetime, timedelta
import os
from tqdm import tqdm
import io
from PIL import Image
import numpy as np
import requests

# NASA GIBS WMS endpoint for EPSG:4326 (geographic coordinates)
wms_url = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi?"

# Define relevant layers for extreme weather prediction
# layers = [
#     "IMERG_Precipitation_Rate",
#     "VIIRS_SNPP_Ice_Surface_Temp_Day",
#     "VIIRS_SNPP_Ice_Surface_Temp_Night",
#     "GHRSST_L4_MUR_Sea_Surface_Temperature",
#     "VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11",
#     "VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR",
#     "CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Day_CH3"
# ]

layers = ["MODIS_Terra_CorrectedReflectance_TrueColor"]

img_size = (1024, 1024)
output_dir = "satellite_images"
os.makedirs(output_dir, exist_ok=True)
metadata_records = []
margin = 10

def is_image_empty(img_data):
    img = Image.open(io.BytesIO(img_data))
    img_array = np.array(img)
    return np.all(img_array == 0)

def get_wms_image(layer, bbox, date):
    params = {
        'SERVICE': 'WMS',
        'VERSION': '1.3.0',
        'REQUEST': 'GetMap',
        'LAYERS': layer,
        'STYLES': '',
        'CRS': 'EPSG:4326',
        'BBOX': ','.join(map(str, bbox)),
        'WIDTH': img_size[0],
        'HEIGHT': img_size[1],
        'FORMAT': 'image/png',
        'TIME': date
    }
    response = requests.get(wms_url, params=params)
    if response.status_code == 200 and response.headers['Content-Type'] == 'image/png':
        return response.content
    else:
        print("Failed to retrieve image: {response.status_code}")
        raise Exception(f"Failed to retrieve image: {response.status_code}")

# Assuming df is your DataFrame with event data
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing events"):
    min_lon = min(row["begin_lon"], row["end_lon"]) - margin
    max_lon = max(row["begin_lon"], row["end_lon"]) + margin
    min_lat = min(row["begin_lat"], row["end_lat"]) - margin
    max_lat = max(row["begin_lat"], row["end_lat"]) + margin
    bbox = [min_lat, min_lon, max_lat, max_lon]
    
    event_date = datetime.strptime(row['begin_date_utc'], "%Y-%m-%d").date()
    
    for days_before in range(3):  # 0, 1, 2 days before the event
        current_date = event_date - timedelta(days=days_before)
        date_str = current_date.strftime("%Y-%m-%d")
        
        for layer in layers:
            safe_event_type = row["event_type"].replace(" ", "_")
            safe_layer_name = layer.replace(" ", "_")
            filename = os.path.join(output_dir, f"event{idx}_{safe_event_type}_{safe_layer_name}_{date_str}.png")
            
            try:
                img_data = get_wms_image(layer, bbox, date_str)
                if is_image_empty(img_data):
                    continue
                
                with open(filename, "wb") as f:
                    f.write(img_data)
               
                
                metadata_records.append({
                    "event_id": idx,
                    "event_type": row["event_type"],
                    "layer": layer,
                    "image_filename": os.path.basename(filename),
                    "download_time": datetime.utcnow().strftime("%Y-%m-%d"),
                    "image_date": date_str,
                    "days_before_event": days_before,
                    "bbox": bbox,
                    "event_date": event_date.strftime("%Y-%m-%d")
                })
            except Exception as e:
                print(f"Failed to download {layer} for event {idx} on {date_str}: {e}")

metadata_df = pd.DataFrame(metadata_records)
metadata_csv = os.path.join(output_dir, "satellite_images_metadata.csv")
metadata_df.to_csv(metadata_csv, index=False)
print(f"Saved metadata to {metadata_csv}")


Processing events:   0%|          | 4/35137 [00:22<55:53:10,  5.73s/it]


KeyboardInterrupt: 

In [33]:
from owslib.wms import WebMapService
from datetime import datetime, timedelta

def get_layer_info(layer_name, wms_url):
    wms = WebMapService(wms_url, version='1.3.0')
    layer = wms.contents[layer_name]
    
    time_info = {}
    if 'time' in layer.dimensions:
        time_dim = layer.dimensions['time']
        if time_dim['values']:
            time_range = time_dim['values'][0].split('/')
            if len(time_range) == 3:
                start_date = datetime.strptime(time_range[0], '%Y-%m-%d')
                end_date = datetime.strptime(time_range[1], '%Y-%m-%d')
                interval = time_range[2]
                
                time_info['start_date'] = start_date.strftime('%Y-%m-%d')
                time_info['end_date'] = end_date.strftime('%Y-%m-%d')
                time_info['interval'] = interval
                
                if interval == 'P1D':
                    time_info['resolution'] = 'Daily'
                elif interval == 'P1M':
                    time_info['resolution'] = 'Monthly'
                elif interval == 'PT1H':
                    time_info['resolution'] = 'Hourly'
                else:
                    time_info['resolution'] = f'Custom ({interval})'
    
    return time_info

# Example usage
wms_url = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"
layers = [
    "MERRA2_2m_Air_Temperature_Monthly",
    "IMERG_Precipitation_Rate",
    "MERRA2_Relative_Humidity_After_Moist_700hPa_Monthly",
    "MERRA2_Surface_Wind_Speed_Monthly",
    "MODIS_Aqua_Cloud_Fraction_Day",
    "MODIS_Terra_Aerosol",
    "MODIS_Aqua_L3_SST_Thermal_9km_Day_Monthly",
    "SMAP_L3_Passive_Enhanced_Day_Soil_Moisture",
    "MODIS_Terra_L3_NDVI_Monthly"
]

for layer_name in layers:
    info = get_layer_info(layer_name, wms_url)
    print(f"Layer: {layer_name}")
    if info:
        print(f"  Resolution: {info['resolution']}")
        print(f"  Date Range: {info['start_date']} to {info['end_date']}")
    else:
        print("  No time information available")
    print()


Layer: MERRA2_2m_Air_Temperature_Monthly
  Resolution: Monthly
  Date Range: 1980-01-01 to 2023-11-01

Layer: IMERG_Precipitation_Rate
  Resolution: Daily
  Date Range: 2000-06-01 to 2025-03-04

Layer: MERRA2_Relative_Humidity_After_Moist_700hPa_Monthly
  Resolution: Monthly
  Date Range: 1980-01-01 to 2023-11-01

Layer: MERRA2_Surface_Wind_Speed_Monthly
  Resolution: Monthly
  Date Range: 1980-01-01 to 2023-11-01

Layer: MODIS_Aqua_Cloud_Fraction_Day
  Resolution: Daily
  Date Range: 2002-07-03 to 2002-07-30

Layer: MODIS_Terra_Aerosol
  Resolution: Daily
  Date Range: 2000-02-24 to 2000-08-06

Layer: MODIS_Aqua_L3_SST_Thermal_9km_Day_Monthly
  Resolution: Monthly
  Date Range: 2002-07-01 to 2022-10-01

Layer: SMAP_L3_Passive_Enhanced_Day_Soil_Moisture
  Resolution: Daily
  Date Range: 2015-03-31 to 2015-05-12

Layer: MODIS_Terra_L3_NDVI_Monthly
  Resolution: Monthly
  Date Range: 2000-03-01 to 2025-01-01



In [37]:
from owslib.wms import WebMapService
from datetime import datetime

# Define the WMS URL
wms_url = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"
wms = WebMapService(wms_url, version="1.3.0")

# Define the years to check for data
years_to_check = [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]

# Function to check if daily or hourly data is available for all specified years
def check_data_availability(layer_name):
    layer = wms.contents[layer_name]
    if 'time' in layer.dimensions:
        time_dim = layer.dimensions['time']
        if time_dim['values']:
            time_range = time_dim['values'][0].split('/')
            if len(time_range) == 3:
                start_date = datetime.strptime(time_range[0][:10], '%Y-%m-%d')
                end_date = datetime.strptime(time_range[1][:10], '%Y-%m-%d')
                interval = time_range[2]
                if interval in ['P1D', 'PT1H']:  # Check if interval is daily or hourly
                    if all(start_date.year <= year <= end_date.year for year in years_to_check):
                        return interval
    return None

# Check each layer and print those with daily or hourly data for all specified years
print("Layers with daily or hourly data available for all specified years:")
for layer_name in wms.contents:
    availability = check_data_availability(layer_name)
    if availability:
        print(f"{layer_name}: {'Daily' if availability == 'P1D' else 'Hourly'}")


Layers with daily or hourly data available for all specified years:
VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11: Daily
VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR: Daily
CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Day_CH3: Daily
CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Difference_Day_CH2_CH3: Daily
CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Difference_Day_CH1_CH3: Daily
CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Difference_Night_CH2_CH3: Daily
CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Difference_Night_CH1_CH3: Daily
CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Night_CH3: Daily
VIIRS_CrIS_SNPP_BT_Band33_Fusion_Day: Daily
VIIRS_CrIS_SNPP_BT_Band33_Fusion_Night: Daily
MOPITT_CO_Daily_Surface_Mixing_Ratio_Day: Daily
MOPITT_CO_Daily_Total_Column_Day: Daily
MOPITT_CO_Daily_Surface_Mixing_Ratio_Night: Daily
MOPITT_CO_Daily_Total_Column_Night: Daily
VIIRS_SNPP_Ice_Surface_Temp_Day: Daily
VIIRS_SNPP_Ice_Surface

In [39]:
final_ds["event_type"].unique()

array(['Flood', 'Heavy Rain', 'Flash Flood', 'Debris Flow', 'Lightning',
       'Thunderstorm Wind', 'Marine High Wind', 'Hail', 'Funnel Cloud',
       'Marine Strong Wind'], dtype=object)

In [1]:
import pandas as pd
df = pd.read_csv("final_ds.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35137 entries, 0 to 35136
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   event_type        35137 non-null  object 
 1   begin_date_time   35137 non-null  object 
 2   cz_timezone       35137 non-null  object 
 3   end_date_time     35137 non-null  object 
 4   begin_lat         35137 non-null  float64
 5   begin_lon         35137 non-null  float64
 6   end_lat           35137 non-null  float64
 7   end_lon           35137 non-null  float64
 8   extreme           35137 non-null  int64  
 9   begin_date_utc    35137 non-null  object 
 10  begin_time_utc    35137 non-null  object 
 11  end_date_utc      35137 non-null  object 
 12  end_time_utc      35137 non-null  object 
 13  cluster           35137 non-null  int64  
 14  start_date_72h    35137 non-null  object 
 15  end_date          35137 non-null  object 
 16  event_datetime    35137 non-null  object

In [2]:
df["extreme"].value_counts()

extreme
0    31678
1     3459
Name: count, dtype: int64

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35137 entries, 0 to 35136
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   event_type        35137 non-null  object 
 1   begin_date_time   35137 non-null  object 
 2   cz_timezone       35137 non-null  object 
 3   end_date_time     35137 non-null  object 
 4   begin_lat         35137 non-null  float64
 5   begin_lon         35137 non-null  float64
 6   end_lat           35137 non-null  float64
 7   end_lon           35137 non-null  float64
 8   extreme           35137 non-null  int64  
 9   begin_date_utc    35137 non-null  object 
 10  begin_time_utc    35137 non-null  object 
 11  end_date_utc      35137 non-null  object 
 12  end_time_utc      35137 non-null  object 
 13  cluster           35137 non-null  int64  
 14  start_date_72h    35137 non-null  object 
 15  end_date          35137 non-null  object 
 16  event_datetime    35137 non-null  object

In [1]:
import pandas as pd
df = pd.read_csv("final_ds.csv")
df = df.iloc[:7000, :]

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   event_type        7000 non-null   object 
 1   begin_date_time   7000 non-null   object 
 2   cz_timezone       7000 non-null   object 
 3   end_date_time     7000 non-null   object 
 4   begin_lat         7000 non-null   float64
 5   begin_lon         7000 non-null   float64
 6   end_lat           7000 non-null   float64
 7   end_lon           7000 non-null   float64
 8   extreme           7000 non-null   int64  
 9   begin_date_utc    7000 non-null   object 
 10  begin_time_utc    7000 non-null   object 
 11  end_date_utc      7000 non-null   object 
 12  end_time_utc      7000 non-null   object 
 13  cluster           7000 non-null   int64  
 14  start_date_72h    7000 non-null   object 
 15  end_date          7000 non-null   object 
 16  event_datetime    7000 non-null   object 


In [1]:
import os
import pickle
import pandas as pd
import numpy as np
from PIL import Image
import io
import requests
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing
from tqdm import tqdm

# Load the dataset
df = pd.read_csv("final_ds.csv")
df = df.iloc[:7000, :]
# NASA GIBS WMS endpoint for EPSG:4326
wms_url = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi?"

# Define relevant layers for extreme weather prediction
layers = [
    "IMERG_Precipitation_Rate",
    "VIIRS_SNPP_Ice_Surface_Temp_Day",
    "VIIRS_SNPP_Ice_Surface_Temp_Night",
    "GHRSST_L4_MUR_Sea_Surface_Temperature",
    "VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11",
    "VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR",
    "CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Day_CH3",
    "MODIS_Terra_CorrectedReflectance_TrueColor"
]

# Image size and output directory
img_size = (512, 512)
output_dir = "satellite_images"
os.makedirs(output_dir, exist_ok=True)
margin = 3

# Checkpoint file to track processed events
checkpoint_file = "processed_events.pkl"
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "rb") as f:
        processed_events = pickle.load(f)
else:
    processed_events = set()

def is_image_empty(img_data):
    """Check if the downloaded image is empty (all pixels are the same)."""
    img = Image.open(io.BytesIO(img_data))
    img_array = np.array(img)
    return np.all(img_array == 0) or np.all(img_array == 255)

def get_wms_image(layer, bbox, date):
    """Retrieve an image from the WMS service for a specific layer, bounding box, and date."""
    params = {
        'SERVICE': 'WMS',
        'VERSION': '1.3.0',
        'REQUEST': 'GetMap',
        'LAYERS': layer,
        'STYLES': '',
        'CRS': 'EPSG:4326',
        'BBOX': ','.join(map(str, bbox)),
        'WIDTH': img_size[0],
        'HEIGHT': img_size[1],
        'FORMAT': 'image/png',
        'TIME': date,
        'TRANSPARENT': False
    }
    response = requests.get(wms_url, params=params)
    if response.status_code == 200 and response.headers['Content-Type'] == 'image/png':
        return response.content
    else:
        return None

def process_event(idx, row, processed_events):
    """
    Process a single event row:
      - Calculate the bounding box with margin.
      - For each of 3 days (0, 1, 2 days before the event), and for each layer,
        download the image, check if it's empty, save it, and record metadata.
    """
    if idx in processed_events:
        return []  # Skip already processed events

    local_metadata = []
    # Compute the bounding box based on event coordinates
    min_lon = min(row["begin_lon"], row["end_lon"]) - margin
    max_lon = max(row["begin_lon"], row["end_lon"]) + margin
    min_lat = min(row["begin_lat"], row["end_lat"]) - margin
    max_lat = max(row["begin_lat"], row["end_lat"]) + margin
    bbox = [min_lat, min_lon, max_lat, max_lon]

    event_date = datetime.strptime(row['begin_date_utc'], "%Y-%m-%d").date()
    
    # Loop through the three days before the event
    for days_before in range(3):
        current_date = event_date - timedelta(days=days_before)
        date_str = current_date.strftime("%Y-%m-%d")

        for layer in layers:
            safe_event_type = row["event_type"].replace(" ", "_")
            safe_layer_name = layer.replace(" ", "_")
            filename = os.path.join(output_dir, f"event{idx}_{safe_event_type}_{safe_layer_name}_{date_str}.png")
            try:
                img_data = get_wms_image(layer, bbox, date_str)
                if img_data is None or is_image_empty(img_data):
                    continue
                with open(filename, "wb") as f:
                    f.write(img_data)
                local_metadata.append({
                    "event_id": idx,
                    "event_type": row["event_type"],
                    "layer": layer,
                    "image_filename": os.path.basename(filename),
                    "download_time": datetime.utcnow().strftime("%Y-%m-%d"),
                    "image_date": date_str,
                    "days_before_event": days_before,
                    "bbox": bbox,
                    "event_date": event_date.strftime("%Y-%m-%d")
                })
            except Exception as e:
                print(f"Error processing event {idx}, layer {layer}, date {date_str}: {e}")
                continue

    # After successfully processing the event
    processed_events.add(idx)
    with open(checkpoint_file, "wb") as f:
        pickle.dump(processed_events, f)
    
    return local_metadata

# Main execution block
all_metadata = []
num_cores = multiprocessing.cpu_count()

# Use ThreadPoolExecutor for parallel processing of events
with ThreadPoolExecutor(max_workers=num_cores) as executor:
    futures = {
        executor.submit(process_event, idx, row, processed_events): idx
        for idx, row in df.iterrows() if idx not in processed_events
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing events"):
        try:
            result = future.result()
            all_metadata.extend(result)
        except Exception as exc:
            print(f"Event processing generated an exception: {exc}")

# Combine new metadata with existing metadata, if any
metadata_csv = os.path.join(output_dir, "satellite_images_metadata.csv")
if os.path.exists(metadata_csv):
    existing_metadata_df = pd.read_csv(metadata_csv)
    metadata_df = pd.DataFrame(all_metadata)
    combined_metadata_df = pd.concat([existing_metadata_df, metadata_df], ignore_index=True)
else:
    combined_metadata_df = pd.DataFrame(all_metadata)

# Save combined metadata to CSV
combined_metadata_df.to_csv(metadata_csv, index=False)
print(f"Saved metadata to {metadata_csv}")


Processing events:  64%|██████▍   | 4464/7000 [2:11:26<46:29,  1.10s/it]   

Error processing event 4468, layer VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR, date 2019-09-19: HTTPSConnectionPool(host='gibs.earthdata.nasa.gov', port=443): Max retries exceeded with url: /wms/epsg4326/best/wms.cgi?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetMap&LAYERS=VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR&STYLES=&CRS=EPSG%3A4326&BBOX=39.88%2C-101.3%2C45.88%2C-95.26&WIDTH=512&HEIGHT=512&FORMAT=image%2Fpng&TIME=2019-09-19&TRANSPARENT=False (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10ced67a0>: Failed to resolve 'gibs.earthdata.nasa.gov' ([Errno 8] nodename nor servname provided, or not known)"))


Processing events:  67%|██████▋   | 4673/7000 [2:17:09<39:05,  1.01s/it]  

Error processing event 4674, layer VIIRS_SNPP_Ice_Surface_Temp_Day, date 2015-11-26: HTTPSConnectionPool(host='gibs.earthdata.nasa.gov', port=443): Max retries exceeded with url: /wms/epsg4326/best/wms.cgi?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetMap&LAYERS=VIIRS_SNPP_Ice_Surface_Temp_Day&STYLES=&CRS=EPSG%3A4326&BBOX=33.86%2C-91.4%2C39.92%2C-85.36&WIDTH=512&HEIGHT=512&FORMAT=image%2Fpng&TIME=2015-11-26&TRANSPARENT=False (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10cf91090>: Failed to resolve 'gibs.earthdata.nasa.gov' ([Errno 8] nodename nor servname provided, or not known)"))


Processing events: 100%|██████████| 7000/7000 [3:31:52<00:00,  1.82s/it]  


Saved metadata to satellite_images/satellite_images_metadata.csv


In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from PIL import Image
import io
import requests
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import multiprocessing
from tqdm import tqdm

# Load the dataset
df = pd.read_csv("final_ds.csv")
df = df.iloc[21000:27999, :]
# NASA GIBS WMS endpoint for EPSG:4326
wms_url = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi?"

# Define relevant layers for extreme weather prediction
layers = [
    "IMERG_Precipitation_Rate",
    "VIIRS_SNPP_Ice_Surface_Temp_Day",
    "VIIRS_SNPP_Ice_Surface_Temp_Night",
    "GHRSST_L4_MUR_Sea_Surface_Temperature",
    "VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11",
    "VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR",
    "CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Day_CH3",
    "MODIS_Terra_CorrectedReflectance_TrueColor"
]

# Image size and output directory
img_size = (512, 512)
output_dir = "satellite_images"
os.makedirs(output_dir, exist_ok=True)
margin = 3

# Checkpoint file to track processed events
checkpoint_file = "processed_events.pkl"
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, "rb") as f:
        processed_events = pickle.load(f)
else:
    processed_events = set()

def is_image_empty(img_data):
    """Check if the downloaded image is empty (all pixels are the same)."""
    img = Image.open(io.BytesIO(img_data))
    img_array = np.array(img)
    return np.all(img_array == 0) or np.all(img_array == 255)

def get_wms_image(layer, bbox, date):
    """Retrieve an image from the WMS service for a specific layer, bounding box, and date."""
    params = {
        'SERVICE': 'WMS',
        'VERSION': '1.3.0',
        'REQUEST': 'GetMap',
        'LAYERS': layer,
        'STYLES': '',
        'CRS': 'EPSG:4326',
        'BBOX': ','.join(map(str, bbox)),
        'WIDTH': img_size[0],
        'HEIGHT': img_size[1],
        'FORMAT': 'image/png',
        'TIME': date,
        'TRANSPARENT': False
    }
    response = requests.get(wms_url, params=params)
    if response.status_code == 200 and response.headers['Content-Type'] == 'image/png':
        return response.content
    else:
        return None

def process_event(idx, row, processed_events):
    """
    Process a single event row:
      - Calculate the bounding box with margin.
      - For each of 3 days (0, 1, 2 days before the event), and for each layer,
        download the image, check if it's empty, save it, and record metadata.
    """
    if idx in processed_events:
        return []  # Skip already processed events

    local_metadata = []
    # Compute the bounding box based on event coordinates
    min_lon = min(row["begin_lon"], row["end_lon"]) - margin
    max_lon = max(row["begin_lon"], row["end_lon"]) + margin
    min_lat = min(row["begin_lat"], row["end_lat"]) - margin
    max_lat = max(row["begin_lat"], row["end_lat"]) + margin
    bbox = [min_lat, min_lon, max_lat, max_lon]

    event_date = datetime.strptime(row['begin_date_utc'], "%Y-%m-%d").date()
    
    # Loop through the three days before the event
    for days_before in range(3):
        current_date = event_date - timedelta(days=days_before)
        date_str = current_date.strftime("%Y-%m-%d")

        for layer in layers:
            safe_event_type = row["event_type"].replace(" ", "_")
            safe_layer_name = layer.replace(" ", "_")
            filename = os.path.join(output_dir, f"event{idx}_{safe_event_type}_{safe_layer_name}_{date_str}.png")
            try:
                img_data = get_wms_image(layer, bbox, date_str)
                if img_data is None or is_image_empty(img_data):
                    continue
                with open(filename, "wb") as f:
                    f.write(img_data)
                local_metadata.append({
                    "event_id": idx,
                    "event_type": row["event_type"],
                    "layer": layer,
                    "image_filename": os.path.basename(filename),
                    "download_time": datetime.utcnow().strftime("%Y-%m-%d"),
                    "image_date": date_str,
                    "days_before_event": days_before,
                    "bbox": bbox,
                    "event_date": event_date.strftime("%Y-%m-%d")
                })
            except Exception as e:
                print(f"Error processing event {idx}, layer {layer}, date {date_str}: {e}")
                continue

    # After successfully processing the event
    processed_events.add(idx)
    with open(checkpoint_file, "wb") as f:
        pickle.dump(processed_events, f)
    
    return local_metadata

# Main execution block
all_metadata = []
num_cores = multiprocessing.cpu_count()

# Use ThreadPoolExecutor for parallel processing of events
with ThreadPoolExecutor(max_workers=num_cores) as executor:
    futures = {
        executor.submit(process_event, idx, row, processed_events): idx
        for idx, row in df.iterrows() if idx not in processed_events
    }
    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing events"):
        try:
            result = future.result()
            all_metadata.extend(result)
        except Exception as exc:
            print(f"Event processing generated an exception: {exc}")

# Combine new metadata with existing metadata, if any
metadata_csv = os.path.join(output_dir, "satellite_images_metadata.csv")
if os.path.exists(metadata_csv):
    existing_metadata_df = pd.read_csv(metadata_csv)
    metadata_df = pd.DataFrame(all_metadata)
    combined_metadata_df = pd.concat([existing_metadata_df, metadata_df], ignore_index=True)
else:
    combined_metadata_df = pd.DataFrame(all_metadata)

# Save combined metadata to CSV
combined_metadata_df.to_csv(metadata_csv, index=False)
print(f"Saved metadata to {metadata_csv}")


Processing events:   6%|▋         | 453/6999 [15:33<2:43:39,  1.50s/it]

Error processing event 21456, layer MODIS_Terra_CorrectedReflectance_TrueColor, date 2013-06-29: HTTPSConnectionPool(host='gibs.earthdata.nasa.gov', port=443): Max retries exceeded with url: /wms/epsg4326/best/wms.cgi?SERVICE=WMS&VERSION=1.3.0&REQUEST=GetMap&LAYERS=MODIS_Terra_CorrectedReflectance_TrueColor&STYLES=&CRS=EPSG%3A4326&BBOX=38.34%2C-78.56%2C44.34%2C-72.52&WIDTH=512&HEIGHT=512&FORMAT=image%2Fpng&TIME=2013-06-29&TRANSPARENT=False (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x10da4d120>: Failed to resolve 'gibs.earthdata.nasa.gov' ([Errno 8] nodename nor servname provided, or not known)"))


Processing events:   7%|▋         | 495/6999 [16:39<3:38:47,  2.02s/it]


In [1]:
import os

folder_path = "satellite_images"  # Replace with your folder path
file_count = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
print(f"Number of files in {folder_path}: {file_count}")


Number of files in satellite_images: 180101


## Integrating final dataset

In [3]:
import pandas as pd
metdata = pd.read_csv("satellite_images/satellite_images_metadata.csv")  
metdata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 167993 entries, 0 to 167992
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   event_id           167993 non-null  int64 
 1   event_type         167993 non-null  object
 2   layer              167993 non-null  object
 3   image_filename     167993 non-null  object
 4   download_time      167993 non-null  object
 5   image_date         167993 non-null  object
 6   days_before_event  167993 non-null  int64 
 7   bbox               167993 non-null  object
 8   event_date         167993 non-null  object
dtypes: int64(2), object(7)
memory usage: 11.5+ MB


In [7]:
metdata.head()

,event_id,event_type,layer,image_filename,download_time,image_date,days_before_event,bbox,event_date
0,4,Flash Flood,IMERG_Precipitation_Rate,event4_Flash_Flood_IMERG_Precipitation_Rate_20...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27
1,4,Flash Flood,VIIRS_SNPP_Ice_Surface_Temp_Day,event4_Flash_Flood_VIIRS_SNPP_Ice_Surface_Temp...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27
2,4,Flash Flood,VIIRS_SNPP_Ice_Surface_Temp_Night,event4_Flash_Flood_VIIRS_SNPP_Ice_Surface_Temp...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27
3,4,Flash Flood,GHRSST_L4_MUR_Sea_Surface_Temperature,event4_Flash_Flood_GHRSST_L4_MUR_Sea_Surface_T...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27
4,4,Flash Flood,VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11,event4_Flash_Flood_VIIRS_SNPP_Cirrus_Reflectan...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27


In [14]:
from PIL import Image
import numpy as np
import io
from tqdm import tqdm
def is_image_empty(img_data):
    """Check if the downloaded image is empty (all pixels are the same)."""
    img = Image.open(io.BytesIO(img_data))
    img_array = np.array(img)
    return np.all(img_array == 0) or np.all(img_array == 255)

In [18]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import io
from tqdm import tqdm

# Define relevant layers for extreme weather prediction
layers = [
    "IMERG_Precipitation_Rate",
    "VIIRS_SNPP_Ice_Surface_Temp_Day",
    "VIIRS_SNPP_Ice_Surface_Temp_Night",
    "GHRSST_L4_MUR_Sea_Surface_Temperature",
    "VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11",
    "VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR",
    "CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Day_CH3",
    "MODIS_Terra_CorrectedReflectance_TrueColor"
]

# Function to check if an image is empty (fully black or white)
def is_image_empty(img_path):
    """Check if an image is empty (all pixels are either 0 or 255)."""
    try:
        with Image.open(img_path) as img:
            img_array = np.array(img)
            return np.all(img_array == 0) or np.all(img_array == 255)
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return True  # Treat unreadable images as empty

# Load the final dataset
final_dataset = pd.read_csv("final_ds.csv")  # Replace with your actual dataset filename

# Folder where images are stored
image_folder = "satellite_images"  # Replace with your actual image folder

# Count empty image rows
empty_rows_count = 0
valid_image_counts = []  # Store number of valid images for each row

for _, row in tqdm(final_dataset.iterrows(), total=len(final_dataset)):
    empty_count = 0
    valid_count = 0
    
    for layer in layers:
        image_filename = f"event{row.name}_{row['event_type']}_{layer}.png"  # Modify based on your naming convention
        image_path = os.path.join(image_folder, image_filename)
        
        if os.path.isfile(image_path):
            if is_image_empty(image_path):
                empty_count += 1
            else:
                valid_count += 1
        else:
            empty_count += 1  # If file is missing, treat it as empty
    
    if valid_count == 0:  # If all images are empty for this row
        empty_rows_count += 1
    else:
        valid_image_counts.append(valid_count)

# Print results
print(f"Total rows where all images are empty: {empty_rows_count}")
print(f"Distribution of valid images in remaining rows:")
print(pd.Series(valid_image_counts).value_counts().sort_index())

100%|██████████| 35137/35137 [00:03<00:00, 10295.51it/s]

Total rows where all images are empty: 35137
Distribution of valid images in remaining rows:
Series([], Name: count, dtype: int64)


In [16]:
import os
import pandas as pd
from PIL import Image
import numpy as np
import io
from tqdm import tqdm

# Function to check if an image is empty (all pixels are either 0 or 255)
def is_image_empty(img_path):
    """Check if the image is empty (all pixels are the same)."""
    try:
        with Image.open(img_path) as img:
            img_array = np.array(img)
            return np.all(img_array == 0) or np.all(img_array == 255)
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return True  # Consider unreadable images as empty

# Load the metadata DataFrame
df = pd.read_csv("satellite_images/satellite_images_metadata.csv")  # Replace with the actual file if different

# Folder where images are stored
image_folder = "satellite_images"  # Replace with your actual image folder

# Iterate through the DataFrame and count empty images
empty_count = 0
total_count = 0

for filename in tqdm(df["image_filename"]):
    image_path = os.path.join(image_folder, filename)
    
    if os.path.isfile(image_path):
        total_count += 1
        if is_image_empty(image_path):
            empty_count += 1
    else:
        print(f"File not found: {image_path}")

print(f"Total valid images checked: {total_count}")
print(f"Number of empty images: {empty_count}")

100%|██████████| 167993/167993 [04:56<00:00, 565.76it/s]

Total valid images checked: 167993
Number of empty images: 0


In [21]:
df.iloc[0,:]["image_filename"]

'event4_Flash_Flood_IMERG_Precipitation_Rate_2014-09-27.png'

In [22]:
final_dataset.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date,event_datetime,prev_72h_weather
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,17:30:00,2012-10-30,03:30:00,4,2012-10-26,2012-10-29,2012-10-29 17:30:00+00:00,[{'date': Timestamp('2012-10-26 18:00:00+0000'...
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,19:00:00,2014-05-16,22:00:00,4,2014-05-12,2014-05-15,2014-05-15 19:00:00+00:00,[{'date': Timestamp('2014-05-12 19:00:00+0000'...
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,15:00:00,2016-10-09,16:00:00,4,2016-10-05,2016-10-08,2016-10-08 15:00:00+00:00,[{'date': Timestamp('2016-10-05 15:00:00+0000'...
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,06:14:00,2014-09-22,09:00:00,19,2014-09-18,2014-09-21,2014-09-22 06:14:00+00:00,[{'date': Timestamp('2014-09-19 07:00:00+0000'...
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,23:00:00,2014-09-28,06:00:00,48,2014-09-24,2014-09-27,2014-09-27 23:00:00+00:00,[{'date': Timestamp('2014-09-24 23:00:00+0000'...


In [ ]:
import os
from tqdm import tqdm

# Function to check if an image is empty (all pixels are either 0 or 255)
def is_image_empty(img_path, threshold=0.70):
    """
    Check if an image is empty or non-informative.
    Args:
        img_path: Path to the image file
        threshold: Percentage of same-colored pixels to consider image empty (0.0 to 1.0)
    Returns:
        bool: True if image is considered empty/non-informative
    """
    try:
        with Image.open(img_path) as img:
            # Convert to grayscale to simplify analysis
            gray_img = img.convert('L')
            img_array = np.array(gray_img)
            
            # Check if image is predominantly white
            white_ratio = np.sum(img_array > 250) / img_array.size
            # Check if image is predominantly black
            black_ratio = np.sum(img_array < 5) / img_array.size
            
            return white_ratio > threshold or black_ratio > threshold
            
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return True  # Consider unreadable images as empty
    
def is_image_informative(img_path, threshold=0.05):
    """
    Check if the image contains sufficient non-white (informative) content.
    
    Parameters:
    - img_path: str, path to the image file.
    - threshold: float, proportion of non-white pixels required to consider the image informative.
    
    Returns:
    - bool: True if the image is informative, False otherwise.
    """
    try:
        with Image.open(img_path) as img:
            # Convert image to grayscale
            gray_img = img.convert('L')
            img_array = np.array(gray_img)
            
            # Calculate the number of non-blank pixels (pixels that are not white)
            non_blank_pixels = np.sum(img_array < 255)  # Changed from '== 0' to '< 255'
            total_pixels = img_array.size
            
            # Calculate the proportion of non-blank pixels
            proportion_non_blank = non_blank_pixels / total_pixels
            
            # Determine if the image is informative based on the threshold
            return proportion_non_blank >= threshold
            
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return False  # Consider unreadable images as non-informative
    
# Load the final dataset
# Folder where images are stored
image_folder = "satellite_images"

# Iterate through all files in the folder and count empty images
informative = 0
total_image_count = 0

for filename in tqdm(os.listdir(image_folder)):
    image_path = os.path.join(image_folder, filename)
    if os.path.isfile(image_path):
        total_image_count += 1
        if is_image_empty(image_path):
            informative += 1

print(f"Total images checked: {total_image_count}")
print(f"Number of empty images: {informative}")

  0%|          | 398/180101 [00:00<06:37, 452.04it/s]


KeyboardInterrupt: 

In [30]:
metdata.head()

,event_id,event_type,layer,image_filename,download_time,image_date,days_before_event,bbox,event_date
0,4,Flash Flood,IMERG_Precipitation_Rate,event4_Flash_Flood_IMERG_Precipitation_Rate_20...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27
1,4,Flash Flood,VIIRS_SNPP_Ice_Surface_Temp_Day,event4_Flash_Flood_VIIRS_SNPP_Ice_Surface_Temp...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27
2,4,Flash Flood,VIIRS_SNPP_Ice_Surface_Temp_Night,event4_Flash_Flood_VIIRS_SNPP_Ice_Surface_Temp...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27
3,4,Flash Flood,GHRSST_L4_MUR_Sea_Surface_Temperature,event4_Flash_Flood_GHRSST_L4_MUR_Sea_Surface_T...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27
4,4,Flash Flood,VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11,event4_Flash_Flood_VIIRS_SNPP_Cirrus_Reflectan...,2025-03-05,2014-09-27,0,"[36.54, -113.92, 42.7, -107.88]",2014-09-27


In [3]:
import pandas as pd

# Load the final dataset
final_df = pd.read_csv("final_ds.csv")

# Load the metadata CSV containing image filenames
metadata_df = pd.read_csv("satellite_images/satellite_images_metadata.csv")

# If your final_df does not have an explicit event identifier,
# add one based on its index.
if 'event_id' not in final_df.columns:
    final_df["event_id"] = final_df.index

# Group metadata by event_id and aggregate image filenames into a list.
filenames_by_event = metadata_df.groupby("event_id")["image_filename"].apply(list).reset_index()
filenames_by_event.rename(columns={"image_filename": "filenames"}, inplace=True)
# print(filenames_by_event)
# Merge the aggregated filenames with the final dataset.
merged_df = pd.merge(final_df, filenames_by_event, on="event_id", how="left")
merged_df.head()
# # Optionally, if you don't need the event_id column, you can drop it:
# # merged_df.drop("event_id", axis=1, inplace=True)

# # Save the merged DataFrame to a new CSV
# merged_df.to_csv("final_ds_with_filenames.csv", index=False)

# print("Merged dataset saved as final_ds_with_filenames.csv")


,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date,event_datetime,prev_72h_weather,event_id,filenames
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,17:30:00,2012-10-30,03:30:00,4,2012-10-26,2012-10-29,2012-10-29 17:30:00+00:00,[{'date': Timestamp('2012-10-26 18:00:00+0000'...,0,[event0_Flood_IMERG_Precipitation_Rate_2012-10...
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,19:00:00,2014-05-16,22:00:00,4,2014-05-12,2014-05-15,2014-05-15 19:00:00+00:00,[{'date': Timestamp('2014-05-12 19:00:00+0000'...,1,[event1_Heavy_Rain_IMERG_Precipitation_Rate_20...
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,15:00:00,2016-10-09,16:00:00,4,2016-10-05,2016-10-08,2016-10-08 15:00:00+00:00,[{'date': Timestamp('2016-10-05 15:00:00+0000'...,2,[event2_Heavy_Rain_IMERG_Precipitation_Rate_20...
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,06:14:00,2014-09-22,09:00:00,19,2014-09-18,2014-09-21,2014-09-22 06:14:00+00:00,[{'date': Timestamp('2014-09-19 07:00:00+0000'...,3,[event3_Flash_Flood_IMERG_Precipitation_Rate_2...
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,23:00:00,2014-09-28,06:00:00,48,2014-09-24,2014-09-27,2014-09-27 23:00:00+00:00,[{'date': Timestamp('2014-09-24 23:00:00+0000'...,4,[event4_Flash_Flood_IMERG_Precipitation_Rate_2...


In [2]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\naman\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [51]:
metadata_df.shape

(167993, 9)

In [50]:
final_ds.shape

(35137, 18)

In [4]:
merged_df.shape

(35137, 20)

In [4]:
merged_df[["begin_date_utc","prev_72h_weather"]]

,begin_date_utc,prev_72h_weather
0,2012-10-29,[{'date': Timestamp('2012-10-26 18:00:00+0000'...
1,2014-05-15,[{'date': Timestamp('2014-05-12 19:00:00+0000'...
2,2016-10-08,[{'date': Timestamp('2016-10-05 15:00:00+0000'...
3,2014-09-22,[{'date': Timestamp('2014-09-19 07:00:00+0000'...
4,2014-09-27,[{'date': Timestamp('2014-09-24 23:00:00+0000'...
...,...,...
35132,2012-09-10,[{'date': Timestamp('2012-09-07 07:00:00+0000'...
35133,2012-03-22,[{'date': Timestamp('2012-03-19 02:00:00+0000'...
35134,2012-09-17,[{'date': Timestamp('2012-09-14 22:00:00+0000'...
35135,2012-04-27,[{'date': Timestamp('2012-04-24 03:00:00+0000'...


In [39]:
merged_df.iloc[0,:]["prev_72h_weather"]

"[{'date': Timestamp('2012-10-26 18:00:00+0000', tz='UTC'), 'temperature_2m': 20.732500076293945, 'relative_humidity_2m': 77.13699340820312, 'dew_point_2m': 16.58249855041504, 'apparent_temperature': 20.407318115234375, 'precipitation': 0.0, 'rain': 0.0, 'snowfall': 0.0, 'snow_depth': 0.0, 'weather_code': 2.0, 'pressure_msl': 1019.0, 'surface_pressure': 1018.881591796875, 'cloud_cover': 75.0, 'cloud_cover_low': 65.0, 'cloud_cover_mid': 0.0, 'cloud_cover_high': 71.0, 'et0_fao_evapotranspiration': 0.2809814512729645, 'vapour_pressure_deficit': 0.5593823194503784, 'wind_speed_10m': 18.0, 'wind_speed_100m': 24.535524368286133, 'wind_direction_10m': 73.73973083496094, 'wind_direction_100m': 75.5559310913086, 'wind_gusts_10m': 30.96000099182129, 'soil_temperature_0_to_7cm': 21.88249969482422, 'soil_temperature_7_to_28cm': 19.182498931884766, 'soil_temperature_28_to_100cm': 19.182498931884766, 'soil_temperature_100_to_255cm': 21.682498931884766, 'soil_moisture_0_to_7cm': 0.33799999952316284, 

In [5]:
merged_df.iloc[0,:]["filenames"]

['event0_Flood_IMERG_Precipitation_Rate_2012-10-29.png',
 'event0_Flood_VIIRS_SNPP_Ice_Surface_Temp_Day_2012-10-29.png',
 'event0_Flood_VIIRS_SNPP_Ice_Surface_Temp_Night_2012-10-29.png',
 'event0_Flood_GHRSST_L4_MUR_Sea_Surface_Temperature_2012-10-29.png',
 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11_2012-10-29.png',
 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR_2012-10-29.png',
 'event0_Flood_CALIPSO_Imaging_Infrared_Radiometer_Brightness_Temperature_Day_CH3_2012-10-29.png',
 'event0_Flood_MODIS_Terra_CorrectedReflectance_TrueColor_2012-10-29.png',
 'event0_Flood_IMERG_Precipitation_Rate_2012-10-28.png',
 'event0_Flood_VIIRS_SNPP_Ice_Surface_Temp_Day_2012-10-28.png',
 'event0_Flood_VIIRS_SNPP_Ice_Surface_Temp_Night_2012-10-28.png',
 'event0_Flood_GHRSST_L4_MUR_Sea_Surface_Temperature_2012-10-28.png',
 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11_2012-10-28.png',
 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR_2012-10-28.png',
 'event0_Flood_CALIPSO_Imaging

In [6]:
# Function to check if an image is empty (all pixels are either 0 or 255)
def is_image_empty(img_path, threshold=0.70):
    """
    Check if an image is empty or non-informative.
    Args:
        img_path: Path to the image file
        threshold: Percentage of same-colored pixels to consider image empty (0.0 to 1.0)
    Returns:
        bool: True if image is considered empty/non-informative
    """
    try:
        with Image.open(img_path) as img:
            # Convert to grayscale to simplify analysis
            gray_img = img.convert('L')
            img_array = np.array(gray_img)
            
            # Check if image is predominantly white
            white_ratio = np.sum(img_array > 250) / img_array.size
            # Check if image is predominantly black
            black_ratio = np.sum(img_array < 5) / img_array.size
            
            return white_ratio > threshold or black_ratio > threshold
            
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return True  # Consider unreadable images as empty

def count_empty_images_by_layer(image_folder, layers):
    """
    Count empty images for each layer in the dataset.
    
    Args:
        image_folder: Path to the folder containing images
        layers: List of layer names to analyze
    """
    # Dictionary to store counts of empty images for each layer
    empty_counts = {layer: 0 for layer in layers}
    total_counts = {layer: 0 for layer in layers}
    
    # Iterate through all images in the folder
    for filename in os.listdir(image_folder):
        if filename.endswith('.png'):
            # Find which layer this image belongs to
            for layer in layers:
                if layer in filename:
                    img_path = os.path.join(image_folder, filename)
                    total_counts[layer] += 1
                    
                    if is_image_empty(img_path):
                        empty_counts[layer] += 1
                    break
    
    # Print results sorted by number of empty images
    print("\nEmpty Image Statistics by Layer:")
    print("-" * 50)
    sorted_layers = sorted(layers, key=lambda x: empty_counts[x], reverse=True)
    
    for layer in sorted_layers:
        if total_counts[layer] > 0:
            empty_ratio = (empty_counts[layer] / total_counts[layer]) * 100
            print(f"{layer}:")
            print(f"  Empty images: {empty_counts[layer]}/{total_counts[layer]} ({empty_ratio:.1f}%)")

# Usage
image_folder = "satellite_images"  # Replace with your image folder path
count_empty_images_by_layer(image_folder, layers)

NameError: name 'layers' is not defined

In [7]:
merged_df["extreme"].value_counts()

extreme
0    31678
1     3459
Name: count, dtype: int64

In [55]:
merged_df = merged_df.iloc[:7000,:]

In [ ]:
merged_df

In [54]:
merged_df.columns

Index(['event_type', 'begin_date_time', 'cz_timezone', 'end_date_time',
       'begin_lat', 'begin_lon', 'end_lat', 'end_lon', 'extreme',
       'begin_date_utc', 'begin_time_utc', 'end_date_utc', 'end_time_utc',
       'cluster', 'start_date_72h', 'end_date', 'event_datetime',
       'prev_72h_weather', 'event_id', 'filenames'],
      dtype='object')

In [3]:
import re
import pandas as pd

def func(s):
    if not isinstance(s, str):
        return []  # Return empty list for non-string inputs
    
    dict_pattern = r'\{[^}]+\}'
    dict_matches = re.findall(dict_pattern, s)
    kv_pattern = r"'([^']+)':\s([^,}]+)"
    extracted_data = []

    for dict_str in dict_matches[:10]:  # Limit to first 10 dictionaries
        kv_matches = re.findall(kv_pattern, dict_str)
        extracted_dict = {key: value.strip() for key, value in kv_matches}
        extracted_data.append(extracted_dict)
    
    return extracted_data
result = []
# Read the CSV in chunks
chunk_size = 10000  # Adjust this based on your available memory
for chunk in pd.read_csv("final_ds_with_filenames.csv", chunksize=chunk_size):
    chunk["prev_72h_weather"] = chunk['prev_72h_weather'].apply(func)
    result.append(chunk)
    # Process or save the chunk here
    print("Processed chunk")
merged_df = pd.concat(result, ignore_index=True)
print("All chunks processed")


Processed chunk
Processed chunk
Processed chunk
Processed chunk
All chunks processed


In [4]:
merged_df[["begin_date_utc","prev_72h_weather"]]

,begin_date_utc,prev_72h_weather
0,2012-10-29,[{'date': 'Timestamp('2012-10-26 18:00:00+0000...
1,2014-05-15,[{'date': 'Timestamp('2014-05-12 19:00:00+0000...
2,2016-10-08,[{'date': 'Timestamp('2016-10-05 15:00:00+0000...
3,2014-09-22,[{'date': 'Timestamp('2014-09-19 07:00:00+0000...
4,2014-09-27,[{'date': 'Timestamp('2014-09-24 23:00:00+0000...
...,...,...
35132,2012-09-10,[{'date': 'Timestamp('2012-09-07 07:00:00+0000...
35133,2012-03-22,[{'date': 'Timestamp('2012-03-19 02:00:00+0000...
35134,2012-09-17,[{'date': 'Timestamp('2012-09-14 22:00:00+0000...
35135,2012-04-27,[{'date': 'Timestamp('2012-04-24 03:00:00+0000...


In [5]:
def expand_weather_data(row):
    weather_data = row['prev_72h_weather']
    for key in weather_data[0].keys():
        if key != 'date':
            row[f'prev_72h_{key}'] = [entry[key] for entry in weather_data]
    return row

merged_df = merged_df.apply(expand_weather_data, axis=1)


In [6]:
len(merged_df.columns)

50

In [7]:
import ast

def clean_and_filter_layers(df, column_name='filename'):
    # Step 1: Convert string representation of list to actual list if needed
    def parse_if_string(x):
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except:
                return x
        return x
    
    # Apply parsing if the column contains string representations of lists
    df[column_name] = df[column_name].apply(parse_if_string)
    
    # Step 2: Filter for specific layers
    patterns = [
        'VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11',
        'VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR',
        'MODIS_Terra_CorrectedReflectance_TrueColor'
    ]
    
    # If the column contains lists, we need to filter elements within each list
    def filter_specific_layers(file_list):
        if isinstance(file_list, list):
            return [f for f in file_list if any(pattern in f for pattern in patterns)]
        return file_list
    
    df[column_name] = df[column_name].apply(filter_specific_layers)
    
    # Remove rows where the filtered list is empty
    df = df[df[column_name].apply(lambda x: len(x) > 0 if isinstance(x, list) else True)]
    
    return df

filtered_df = clean_and_filter_layers(merged_df, column_name='filenames')

In [8]:
filtered_df = filtered_df.iloc[:7000,:]

In [9]:
filtered_df.shape

(7000, 50)

In [10]:
from PIL import Image
import numpy as np
from pathlib import Path

# Function to check if an image is empty (all pixels are either 0 or 255)
def is_image_empty(img_path, threshold=0.70):
    """
    Check if an image is empty or non-informative.
    Args:
        img_path: Path to the image file
        threshold: Percentage of same-colored pixels to consider image empty (0.0 to 1.0)
    Returns:
        bool: True if image is considered empty/non-informative
    """
    try:
        with Image.open(img_path) as img:
            # Convert to grayscale to simplify analysis
            gray_img = img.convert('L')
            img_array = np.array(gray_img)
            
            # Check if image is predominantly white
            white_ratio = np.sum(img_array > 250) / img_array.size
            # Check if image is predominantly black
            black_ratio = np.sum(img_array < 5) / img_array.size
            
            return white_ratio > threshold or black_ratio > threshold
            
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return True  # Consider unreadable images as empty

def check_and_filter_empty_images(df, base_path, column_name='filenames', threshold=0.70):
    """
    Check filtered images for emptiness and remove rows containing any empty images.
    
    Args:
        df: DataFrame containing lists of image filenames
        base_path: Base directory path where images are stored
        column_name: Name of the column containing image filenames
        threshold: Threshold for determining empty images
        
    Returns:
        DataFrame with rows removed where any image in the list is empty
    """
    def check_image_list(image_list):
        for img_name in image_list:
            img_path = Path(base_path) / img_name
            if is_image_empty(img_path, threshold):
                return False  # If any image is empty, return False
        return True  # All images are valid
    
    # Apply the check to each row and keep only rows where all images are valid
    mask = df[column_name].apply(check_image_list)
    filtered_df = df[mask]
    
    # Print statistics
    removed_count = len(df) - len(filtered_df)
    print(f"Removed {removed_count} rows containing empty images")
    print(f"Remaining rows: {len(filtered_df)}")
    
    return filtered_df

# Usage example:

In [11]:
# Assuming you have already filtered for the three specific layers
base_path = "satellite_images"  # Replace with your actual path
final_df = check_and_filter_empty_images(filtered_df, base_path)

Removed 136 rows containing empty images
Remaining rows: 6864


In [12]:
final_df["extreme"].value_counts()  

extreme
0    6427
1     437
Name: count, dtype: int64

In [13]:
final_df.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,...,prev_72h_wind_direction_100m,prev_72h_wind_gusts_10m,prev_72h_soil_temperature_0_to_7cm,prev_72h_soil_temperature_7_to_28cm,prev_72h_soil_temperature_28_to_100cm,prev_72h_soil_temperature_100_to_255cm,prev_72h_soil_moisture_0_to_7cm,prev_72h_soil_moisture_7_to_28cm,prev_72h_soil_moisture_28_to_100cm,prev_72h_soil_moisture_100_to_255cm
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,...,"[75.5559310913086, 77.90525817871094, 78.54128...","[30.96000099182129, 33.47999954223633, 34.2000...","[21.88249969482422, 21.932498931884766, 21.482...","[19.182498931884766, 19.38249969482422, 19.482...","[19.182498931884766, 19.182498931884766, 19.18...","[21.682498931884766, 21.682498931884766, 21.68...","[0.33799999952316284, 0.3370000123977661, 0.33...","[0.3540000021457672, 0.3529999852180481, 0.352...","[0.3330000042915344, 0.3319999873638153, 0.331...","[0.3449999988079071, 0.3449999988079071, 0.344..."
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,...,"[198.64962768554688, 193.87754821777344, 191.1...","[39.959999084472656, 40.68000030517578, 40.319...","[24.58249855041504, 24.38249969482422, 23.8824...","[19.58249855041504, 19.782499313354492, 19.932...","[14.982500076293945, 14.982500076293945, 15.03...","[9.782500267028809, 9.782500267028809, 9.78250...","[0.3240000009536743, 0.3230000138282776, 0.321...","[0.3479999899864197, 0.3479999899864197, 0.347...","[0.37299999594688416, 0.37299999594688416, 0.3...","[0.414000004529953, 0.414000004529953, 0.41400..."
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,...,"[55.645606994628906, 56.97612762451172, 57.002...","[47.15999984741211, 45.36000061035156, 44.6399...","[21.032499313354492, 21.782499313354492, 22.23...","[20.932498931884766, 21.032499313354492, 21.13...","[22.732500076293945, 22.732500076293945, 22.73...","[23.08249855041504, 23.08249855041504, 23.0824...","[0.38999998569488525, 0.3889999985694885, 0.38...","[0.4020000100135803, 0.4009999930858612, 0.400...","[0.3880000114440918, 0.3880000114440918, 0.388...","[0.37700000405311584, 0.37700000405311584, 0.3..."
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,...,"[138.81417846679688, 270.0, 311.98712158203125...","[6.479999542236328, 5.399999618530273, 7.19999...","[19.298999786376953, 19.39900016784668, 19.249...","[21.39900016784668, 21.3489990234375, 21.24900...","[25.3489990234375, 25.3489990234375, 25.298999...","[26.198999404907227, 26.198999404907227, 26.19...","[0.3709999918937683, 0.36899998784065247, 0.36...","[0.3070000112056732, 0.3070000112056732, 0.307...","[0.15800000727176666, 0.15800000727176666, 0.1...","[0.15700000524520874, 0.15700000524520874, 0.1..."
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,...,"[173.4803009033203, 169.69520568847656, 168.69...","[26.639999389648438, 24.119998931884766, 19.79...","[30.48900032043457, 28.388999938964844, 25.088...","[19.68899917602539, 19.93899917602539, 20.0889...","[19.48900032043457, 19.48900032043457, 19.4890...","[18.538999557495117, 18.538999557495117, 18.53...","[0.04800000041723251, 0.04699999839067459, 0.0...","[0.16099999845027924, 0.16099999845027924, 0.1...","[0.22300000488758087, 0.22300000488758087, 0.2...","[0.2590000033378601, 0.2590000033378601, 0.259..."


In [35]:
final_df.iloc[1,:]

event_type                                                                       Heavy Rain
begin_date_time                                                         2014-05-15 14:00:00
cz_timezone                                                                           EST-5
end_date_time                                                           2014-05-16 17:00:00
begin_lat                                                                             37.84
begin_lon                                                                            -75.48
end_lat                                                                               37.82
end_lon                                                                              -75.62
extreme                                                                                   0
begin_date_utc                                                                   2014-05-15
begin_time_utc                                                                  

In [41]:
final_df.iloc[2,:]

event_type                                                                       Heavy Rain
begin_date_time                                                         2016-10-08 10:00:00
cz_timezone                                                                           EST-5
end_date_time                                                           2016-10-09 11:00:00
begin_lat                                                                             37.84
begin_lon                                                                            -75.48
end_lat                                                                               37.84
end_lon                                                                              -75.48
extreme                                                                                   0
begin_date_utc                                                                   2016-10-08
begin_time_utc                                                                  

In [16]:
final_df.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,...,prev_72h_wind_direction_100m,prev_72h_wind_gusts_10m,prev_72h_soil_temperature_0_to_7cm,prev_72h_soil_temperature_7_to_28cm,prev_72h_soil_temperature_28_to_100cm,prev_72h_soil_temperature_100_to_255cm,prev_72h_soil_moisture_0_to_7cm,prev_72h_soil_moisture_7_to_28cm,prev_72h_soil_moisture_28_to_100cm,prev_72h_soil_moisture_100_to_255cm
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,...,"[75.5559310913086, 77.90525817871094, 78.54128...","[30.96000099182129, 33.47999954223633, 34.2000...","[21.88249969482422, 21.932498931884766, 21.482...","[19.182498931884766, 19.38249969482422, 19.482...","[19.182498931884766, 19.182498931884766, 19.18...","[21.682498931884766, 21.682498931884766, 21.68...","[0.33799999952316284, 0.3370000123977661, 0.33...","[0.3540000021457672, 0.3529999852180481, 0.352...","[0.3330000042915344, 0.3319999873638153, 0.331...","[0.3449999988079071, 0.3449999988079071, 0.344..."
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,...,"[198.64962768554688, 193.87754821777344, 191.1...","[39.959999084472656, 40.68000030517578, 40.319...","[24.58249855041504, 24.38249969482422, 23.8824...","[19.58249855041504, 19.782499313354492, 19.932...","[14.982500076293945, 14.982500076293945, 15.03...","[9.782500267028809, 9.782500267028809, 9.78250...","[0.3240000009536743, 0.3230000138282776, 0.321...","[0.3479999899864197, 0.3479999899864197, 0.347...","[0.37299999594688416, 0.37299999594688416, 0.3...","[0.414000004529953, 0.414000004529953, 0.41400..."
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,...,"[55.645606994628906, 56.97612762451172, 57.002...","[47.15999984741211, 45.36000061035156, 44.6399...","[21.032499313354492, 21.782499313354492, 22.23...","[20.932498931884766, 21.032499313354492, 21.13...","[22.732500076293945, 22.732500076293945, 22.73...","[23.08249855041504, 23.08249855041504, 23.0824...","[0.38999998569488525, 0.3889999985694885, 0.38...","[0.4020000100135803, 0.4009999930858612, 0.400...","[0.3880000114440918, 0.3880000114440918, 0.388...","[0.37700000405311584, 0.37700000405311584, 0.3..."
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,...,"[138.81417846679688, 270.0, 311.98712158203125...","[6.479999542236328, 5.399999618530273, 7.19999...","[19.298999786376953, 19.39900016784668, 19.249...","[21.39900016784668, 21.3489990234375, 21.24900...","[25.3489990234375, 25.3489990234375, 25.298999...","[26.198999404907227, 26.198999404907227, 26.19...","[0.3709999918937683, 0.36899998784065247, 0.36...","[0.3070000112056732, 0.3070000112056732, 0.307...","[0.15800000727176666, 0.15800000727176666, 0.1...","[0.15700000524520874, 0.15700000524520874, 0.1..."
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,...,"[173.4803009033203, 169.69520568847656, 168.69...","[26.639999389648438, 24.119998931884766, 19.79...","[30.48900032043457, 28.388999938964844, 25.088...","[19.68899917602539, 19.93899917602539, 20.0889...","[19.48900032043457, 19.48900032043457, 19.4890...","[18.538999557495117, 18.538999557495117, 18.53...","[0.04800000041723251, 0.04699999839067459, 0.0...","[0.16099999845027924, 0.16099999845027924, 0.1...","[0.22300000488758087, 0.22300000488758087, 0.2...","[0.2590000033378601, 0.2590000033378601, 0.259..."


In [19]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6864 entries, 0 to 6999
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   event_type                              6864 non-null   object 
 1   begin_date_time                         6864 non-null   object 
 2   cz_timezone                             6864 non-null   object 
 3   end_date_time                           6864 non-null   object 
 4   begin_lat                               6864 non-null   float64
 5   begin_lon                               6864 non-null   float64
 6   end_lat                                 6864 non-null   float64
 7   end_lon                                 6864 non-null   float64
 8   extreme                                 6864 non-null   int64  
 9   begin_date_utc                          6864 non-null   object 
 10  begin_time_utc                          6864 non-null   object 
 

In [17]:
final_df.columns

Index(['event_type', 'begin_date_time', 'cz_timezone', 'end_date_time',
       'begin_lat', 'begin_lon', 'end_lat', 'end_lon', 'extreme',
       'begin_date_utc', 'begin_time_utc', 'end_date_utc', 'end_time_utc',
       'cluster', 'start_date_72h', 'end_date', 'event_datetime',
       'prev_72h_weather', 'event_id', 'filenames', 'prev_72h_temperature_2m',
       'prev_72h_relative_humidity_2m', 'prev_72h_dew_point_2m',
       'prev_72h_apparent_temperature', 'prev_72h_precipitation',
       'prev_72h_rain', 'prev_72h_snowfall', 'prev_72h_snow_depth',
       'prev_72h_weather_code', 'prev_72h_pressure_msl',
       'prev_72h_surface_pressure', 'prev_72h_cloud_cover',
       'prev_72h_cloud_cover_low', 'prev_72h_cloud_cover_mid',
       'prev_72h_cloud_cover_high', 'prev_72h_et0_fao_evapotranspiration',
       'prev_72h_vapour_pressure_deficit', 'prev_72h_wind_speed_10m',
       'prev_72h_wind_speed_100m', 'prev_72h_wind_direction_10m',
       'prev_72h_wind_direction_100m', 'prev_72h_win

In [41]:
import ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

# -------------------------------
# Step 1: Data Preprocessing
# -------------------------------
def preprocess_data(df):
    # Select relevant columns (excluding 'prev_72h_weather')
    feature_columns = [col for col in df.columns if col.startswith('prev_72h_') and col != 'prev_72h_weather']

    def safe_eval(x):
        # If x is not a scalar and is iterable (but not a string), convert to list.
        if not np.isscalar(x) and not isinstance(x, str):
            try:
                return list(x)
            except Exception:
                return []
        # If x is scalar, check for NaN.
        if pd.isna(x):
            return []
        # If x is a string, try to literal_eval it.
        if isinstance(x, str):
            x = x.strip()
            if x == "":
                return []
            try:
                return ast.literal_eval(x)
            except Exception as e:
                print(f"Failed to evaluate '{x}': {e}")
                return []
        # Otherwise, if it's a number, wrap it in a list.
        return [x]

    def extract_numeric(x):
        values = safe_eval(x)
        if not values:
            return [np.nan]
        numeric_vals = []
        for val in values:
            try:
                numeric_vals.append(float(val))
            except Exception:
                numeric_vals.append(np.nan)
        return numeric_vals

    # Apply extraction cellwise using applymap.
    processed = df[feature_columns].applymap(extract_numeric)

    # Expand each cell's list into separate columns.
    expanded_dfs = []
    for col in feature_columns:
        expanded = processed[col].apply(pd.Series)
        # Rename columns, e.g., "prev_72h_temperature_2m_0", "prev_72h_temperature_2m_1", etc.
        expanded = expanded.add_prefix(f"{col}_")
        expanded_dfs.append(expanded)

    # Concatenate all expanded columns horizontally.
    processed_df = pd.concat(expanded_dfs, axis=1)
    # Add target variable
    processed_df['extreme'] = df['extreme']
    
    print("Preprocessed DataFrame head:")
    print(processed_df.head())
    return processed_df

# -------------------------------
# Step 2: Compute Class Weights (if needed)
# -------------------------------
def compute_class_weights(y):
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y), y=y)
    return dict(zip(np.unique(y), class_weights))

# -------------------------------
# Step 3: Define Models and Parameter Grids
# -------------------------------
models = {
    "Random Forest": {
        "estimator": RandomForestClassifier(random_state=42, class_weight='balanced'),
        "param_grid": {
            "n_estimators": [100, 200],
            "max_depth": [5, 10, None],
            "max_features": ["sqrt", "log2"]
        }
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingClassifier(random_state=42),
        "param_grid": {
            "n_estimators": [100, 200],
            "max_depth": [3, 5],
            "learning_rate": [0.05, 0.1]
        }
    },
    "Logistic Regression": {
        "estimator": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
        "param_grid": {
            "C": [0.1, 1, 10],
            "penalty": ["l2"],
            "solver": ["lbfgs"]
        }
    },
    "SVC": {
        "estimator": SVC(probability=True, random_state=42, class_weight='balanced'),
        "param_grid": {
            "C": [0.1, 1, 10],
            "kernel": ["rbf", "linear"],
            "gamma": ["scale", "auto"]
        }
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "param_grid": {
            "n_neighbors": [3, 5, 7],
            "weights": ["uniform", "distance"]
        }
    }
}

# -------------------------------
# Step 4: Define a Function to Evaluate a Model Using TimeSeriesSplit
# -------------------------------
def evaluate_model(estimator, X, y, cv_splits=5):
    tscv = TimeSeriesSplit(n_splits=cv_splits)
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        estimator.fit(X_train, y_train)
        y_pred = estimator.predict(X_val)
        try:
            y_pred_proba = estimator.predict_proba(X_val)[:, 1]
        except AttributeError:
            y_pred_proba = estimator.decision_function(X_val)
        accs.append(accuracy_score(y_val, y_pred))
        precs.append(precision_score(y_val, y_pred, pos_label=1, zero_division=0))
        recs.append(recall_score(y_val, y_pred, pos_label=1, zero_division=0))
        f1s.append(f1_score(y_val, y_pred, pos_label=1, zero_division=0))
        aucs.append(roc_auc_score(y_val, y_pred_proba))
    return {
        "Accuracy": np.mean(accs),
        "Precision": np.mean(precs),
        "Recall": np.mean(recs),
        "F1-score": np.mean(f1s),
        "AUC": np.mean(aucs)
    }

# -------------------------------
# Step 5: Main Script
# -------------------------------
if __name__ == '__main__':
    # Load your DataFrame (adjust this to your data source)
    # For example:
    # df = pd.read_csv("your_data.csv")
    # Here, we assume df is already defined.
    
    print("Original DataFrame shape:", df.shape)
    
    # Preprocess the data
    processed_df = preprocess_data(df)
    
    # Split features and target
    X = processed_df.drop('extreme', axis=1)
    y = processed_df['extreme']
    
    # Handle missing values:
    # Drop columns that are entirely NaN
    X = X.dropna(axis=1, how='all')
    # Impute remaining missing values with the mean
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='mean')
    X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
    
    # Normalize features
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=X_imputed.columns, index=X_imputed.index)
    
    # Optional: Compute and display class weights
    weights = compute_class_weights(y)
    print("Computed class weights:", weights)
    
    # To store final results for each model
    results = []
    
    # Loop over each model: run GridSearchCV with TimeSeriesSplit and evaluate best estimator
    for model_name, model_dict in models.items():
        print(f"\nTuning and evaluating model: {model_name}")
        estimator = model_dict["estimator"]
        param_grid = model_dict["param_grid"]
        tscv = TimeSeriesSplit(n_splits=5)
        grid = GridSearchCV(estimator, param_grid, cv=tscv, scoring='accuracy', n_jobs=-1, error_score='raise')
        try:
            grid.fit(X_scaled, y)
        except Exception as e:
            print(f"GridSearchCV failed for {model_name}: {e}")
            continue
        best_estimator = grid.best_estimator_
        print(f"Best parameters for {model_name}: {grid.best_params_}")
        metrics = evaluate_model(best_estimator, X_scaled, y, cv_splits=5)
        metrics["Model"] = model_name
        results.append(metrics)
    
    # Create a final results DataFrame and display it
    results_df = pd.DataFrame(results).set_index("Model")
    results_df = results_df[["Accuracy", "Precision", "Recall", "F1-score", "AUC"]]
    
    print("\nFinal Model Comparison Report:")
    print(results_df.round(4))


Original DataFrame shape: (6864, 50)


/var/folders/b0/7pq7nl0s6mdbbp1gwm09my6w0000gp/T/ipykernel_56574/1402383168.py:57: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  processed = df[feature_columns].applymap(extract_numeric)


Preprocessed DataFrame head:
   prev_72h_temperature_2m_0  prev_72h_temperature_2m_1  \
0                  20.732500                  20.932499   
1                  23.432499                  23.182499   
2                  19.982500                  20.482500   
3                  18.299000                  17.948999   
4                  25.188999                  24.389000   

   prev_72h_temperature_2m_2  prev_72h_temperature_2m_3  \
0                  20.332499                  19.682499   
1                  22.782499                  22.232500   
2                  20.732500                  20.682499   
3                  17.799000                  17.698999   
4                  22.239000                  20.838999   

   prev_72h_temperature_2m_4  prev_72h_temperature_2m_5  \
0                  18.982500                  18.532499   
1                  21.432499                  20.832499   
2                  20.382500                  19.932499   
3                  17.348

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

Best parameters for Logistic Regression: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Tuning and evaluating model: SVC
Best parameters for SVC: {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}

Tuning and evaluating model: KNN
Best parameters for KNN: {'n_neighbors': 7, 'weights': 'uniform'}

Final Model Comparison Report:
                     Accuracy  Precision  Recall  F1-score     AUC
Model                                                             
Random Forest          0.9448     0.3051  0.0187    0.0319  0.6092
Gradient Boosting      0.9462     0.1333  0.0108    0.0199  0.5829
Logistic Regression    0.6918     0.0525  0.2843    0.0875  0.5129
SVC                    0.8885     0.0752  0.1065    0.0874  0.5579
KNN                    0.9350     0.1103  0.0295    0.0424  0.5283


In [69]:
final_df[final_df["extreme"]==0]

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,...,prev_72h_wind_direction_100m,prev_72h_wind_gusts_10m,prev_72h_soil_temperature_0_to_7cm,prev_72h_soil_temperature_7_to_28cm,prev_72h_soil_temperature_28_to_100cm,prev_72h_soil_temperature_100_to_255cm,prev_72h_soil_moisture_0_to_7cm,prev_72h_soil_moisture_7_to_28cm,prev_72h_soil_moisture_28_to_100cm,prev_72h_soil_moisture_100_to_255cm
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,...,"[75.5559310913086, 77.90525817871094, 78.54128...","[30.96000099182129, 33.47999954223633, 34.2000...","[21.88249969482422, 21.932498931884766, 21.482...","[19.182498931884766, 19.38249969482422, 19.482...","[19.182498931884766, 19.182498931884766, 19.18...","[21.682498931884766, 21.682498931884766, 21.68...","[0.33799999952316284, 0.3370000123977661, 0.33...","[0.3540000021457672, 0.3529999852180481, 0.352...","[0.3330000042915344, 0.3319999873638153, 0.331...","[0.3449999988079071, 0.3449999988079071, 0.344..."
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,...,"[198.64962768554688, 193.87754821777344, 191.1...","[39.959999084472656, 40.68000030517578, 40.319...","[24.58249855041504, 24.38249969482422, 23.8824...","[19.58249855041504, 19.782499313354492, 19.932...","[14.982500076293945, 14.982500076293945, 15.03...","[9.782500267028809, 9.782500267028809, 9.78250...","[0.3240000009536743, 0.3230000138282776, 0.321...","[0.3479999899864197, 0.3479999899864197, 0.347...","[0.37299999594688416, 0.37299999594688416, 0.3...","[0.414000004529953, 0.414000004529953, 0.41400..."
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,...,"[55.645606994628906, 56.97612762451172, 57.002...","[47.15999984741211, 45.36000061035156, 44.6399...","[21.032499313354492, 21.782499313354492, 22.23...","[20.932498931884766, 21.032499313354492, 21.13...","[22.732500076293945, 22.732500076293945, 22.73...","[23.08249855041504, 23.08249855041504, 23.0824...","[0.38999998569488525, 0.3889999985694885, 0.38...","[0.4020000100135803, 0.4009999930858612, 0.400...","[0.3880000114440918, 0.3880000114440918, 0.388...","[0.37700000405311584, 0.37700000405311584, 0.3..."
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,...,"[138.81417846679688, 270.0, 311.98712158203125...","[6.479999542236328, 5.399999618530273, 7.19999...","[19.298999786376953, 19.39900016784668, 19.249...","[21.39900016784668, 21.3489990234375, 21.24900...","[25.3489990234375, 25.3489990234375, 25.298999...","[26.198999404907227, 26.198999404907227, 26.19...","[0.3709999918937683, 0.36899998784065247, 0.36...","[0.3070000112056732, 0.3070000112056732, 0.307...","[0.15800000727176666, 0.15800000727176666, 0.1...","[0.15700000524520874, 0.15700000524520874, 0.1..."
5,Heavy Rain,2014-10-18 09:00:00,CST-6,2014-10-18 12:10:00,26.06,-97.48,26.26,-97.44,0,2014-10-18,...,"[248.1985321044922, 220.2362823486328, 157.380...","[17.280000686645508, 18.0, 16.919998168945312,...","[23.18899917602539, 25.538999557495117, 27.588...","[23.638999938964844, 23.838998794555664, 24.13...","[27.888999938964844, 27.838998794555664, 27.83...","[28.288999557495117, 28.288999557495117, 28.28...","[0.2590000033378601, 0.25699999928474426, 0.25...","[0.27300000190734863, 0.2720000147819519, 0.27...","[0.24699999392032623, 0.24699999392032623, 0.2...","[0.17299999296665192, 0.17299999296665192, 0.1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6995,Heavy Rain,2020-08-03 11:00:00,EST-5,2020-08-04 09:00:00,37.88,-76.42,37.88,-76.42,0,2020-08-03,...,"[270.0, 285.7087097167969, 296.5649719238281, ...","[32.76000213623047, 31.68000030517578, 37.7999...","[26.926498413085938, 27.226499557495117, 28.12...","[26.776498794555664, 26.776498794555664, 26.77...","[24.726499557495117, 24.726499557495

In [68]:
final_df[final_df["extreme"]==1]


,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,...,prev_72h_wind_direction_100m,prev_72h_wind_gusts_10m,prev_72h_soil_temperature_0_to_7cm,prev_72h_soil_temperature_7_to_28cm,prev_72h_soil_temperature_28_to_100cm,prev_72h_soil_temperature_100_to_255cm,prev_72h_soil_moisture_0_to_7cm,prev_72h_soil_moisture_7_to_28cm,prev_72h_soil_moisture_28_to_100cm,prev_72h_soil_moisture_100_to_255cm
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,...,"[173.4803009033203, 169.69520568847656, 168.69...","[26.639999389648438, 24.119998931884766, 19.79...","[30.48900032043457, 28.388999938964844, 25.088...","[19.68899917602539, 19.93899917602539, 20.0889...","[19.48900032043457, 19.48900032043457, 19.4890...","[18.538999557495117, 18.538999557495117, 18.53...","[0.04800000041723251, 0.04699999839067459, 0.0...","[0.16099999845027924, 0.16099999845027924, 0.1...","[0.22300000488758087, 0.22300000488758087, 0.2...","[0.2590000033378601, 0.2590000033378601, 0.259..."
10,Flash Flood,2014-09-21 12:30:00,MST-7,2014-09-21 23:30:00,33.56,-105.58,33.44,-104.62,1,2014-09-21,...,"[157.24899291992188, 164.35769653320312, 224.9...","[20.51999855041504, 20.880001068115234, 19.440...","[19.10249900817871, 18.502500534057617, 18.352...","[16.65250015258789, 16.802499771118164, 16.952...","[17.502500534057617, 17.502500534057617, 17.50...","[17.702499389648438, 17.702499389648438, 17.70...","[0.4050000011920929, 0.4050000011920929, 0.405...","[0.3840000033378601, 0.3840000033378601, 0.384...","[0.1770000010728836, 0.1770000010728836, 0.177...","[0.1589999943971634, 0.1589999943971634, 0.158..."
12,Flash Flood,2014-09-27 09:30:00,MST-7,2014-09-27 20:00:00,37.54,-112.86,37.66,-114.04,1,2014-09-27,...,"[152.1028289794922, 157.47938537597656, 162.71...","[32.39999771118164, 37.07999801635742, 37.7999...","[18.75950050354004, 21.00950050354004, 22.8094...","[14.459500312805176, 14.759500503540039, 15.15...","[15.559499740600586, 15.559499740600586, 15.55...","[14.009500503540039, 14.009500503540039, 14.00...","[0.15399999916553497, 0.15199999511241913, 0.1...","[0.15800000727176666, 0.15800000727176666, 0.1...","[0.18700000643730164, 0.18700000643730164, 0.1...","[0.26100000739097595, 0.26100000739097595, 0.2..."
18,Flash Flood,2014-09-09 17:40:00,CST-6,2014-09-10 05:15:00,41.50,-93.68,41.50,-93.66,1,2014-09-09,...,"[26.564985275268555, 251.56495666503906, 215.5...","[10.799999237060547, 5.039999961853027, 4.6799...","[21.804500579833984, 20.10449981689453, 18.554...","[20.954500198364258, 20.90450096130371, 20.754...","[21.35449981689453, 21.35449981689453, 21.3045...","[17.954500198364258, 17.954500198364258, 17.95...","[0.3059999942779541, 0.3059999942779541, 0.305...","[0.2930000126361847, 0.2930000126361847, 0.293...","[0.2460000067949295, 0.2460000067949295, 0.246...","[0.31700000166893005, 0.31700000166893005, 0.3..."
21,Flash Flood,2014-09-09 18:47:00,CST-6,2014-09-10 05:00:00,40.94,-94.38,40.90,-94.38,1,2014-09-10,...,"[93.17977142333984, 100.78425598144531, 112.24...","[7.559999465942383, 9.359999656677246, 10.0799...","[19.87150001525879, 18.42150115966797, 17.3715...","[20.821500778198242, 20.67150115966797, 20.471...","[21.821500778198242, 21.821500778198242, 21.82...","[18.821500778198242, 18.821500778198242, 18.82...","[0.328000009059906, 0.3269999921321869, 0.3269...","[0.30399999022483826, 0.30399999022483826, 0.3...","[0.2240000069141388, 0.2240000069141388, 0.224...","[0.30300000309944153, 0.30300000309944153, 0.3..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6818,Flash Flood,2020-07-30 11:00:00,AST-4,2020-07-30 21:15:00,18.22,-67.14,18.22,-67.14,1,2020-07-30,...,"[53.84172821044922, 69.4438705444336, 53.34380...","[38.880001068115234, 41.03999710083008, 37.079...","[28.18899917602539, 28.588998794555664, 29.088...","[26.838998794555664, 26.93899917602539, 27.038...","[26.63899993896484

In [43]:
final_df.columns

Index(['event_type', 'begin_date_time', 'cz_timezone', 'end_date_time',
       'begin_lat', 'begin_lon', 'end_lat', 'end_lon', 'extreme',
       'begin_date_utc', 'begin_time_utc', 'end_date_utc', 'end_time_utc',
       'cluster', 'start_date_72h', 'end_date', 'event_datetime',
       'prev_72h_weather', 'event_id', 'filenames', 'prev_72h_temperature_2m',
       'prev_72h_relative_humidity_2m', 'prev_72h_dew_point_2m',
       'prev_72h_apparent_temperature', 'prev_72h_precipitation',
       'prev_72h_rain', 'prev_72h_snowfall', 'prev_72h_snow_depth',
       'prev_72h_weather_code', 'prev_72h_pressure_msl',
       'prev_72h_surface_pressure', 'prev_72h_cloud_cover',
       'prev_72h_cloud_cover_low', 'prev_72h_cloud_cover_mid',
       'prev_72h_cloud_cover_high', 'prev_72h_et0_fao_evapotranspiration',
       'prev_72h_vapour_pressure_deficit', 'prev_72h_wind_speed_10m',
       'prev_72h_wind_speed_100m', 'prev_72h_wind_direction_10m',
       'prev_72h_wind_direction_100m', 'prev_72h_win

In [44]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6864 entries, 0 to 6999
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   event_type                              6864 non-null   object 
 1   begin_date_time                         6864 non-null   object 
 2   cz_timezone                             6864 non-null   object 
 3   end_date_time                           6864 non-null   object 
 4   begin_lat                               6864 non-null   float64
 5   begin_lon                               6864 non-null   float64
 6   end_lat                                 6864 non-null   float64
 7   end_lon                                 6864 non-null   float64
 8   extreme                                 6864 non-null   int64  
 9   begin_date_utc                          6864 non-null   object 
 10  begin_time_utc                          6864 non-null   object 
 

In [49]:
df.drop(columns=["prev_72h_weather"], inplace=True)

In [65]:
# Corrected Implementation for Multi-Class AUC Calculation
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Masking, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_class_weight
from imblearn.pipeline import Pipeline

# Custom Reshaper for Temporal Data
class TemporalReshaper:
    def fit(self, X, y=None): return self
    def transform(self, X): return X.reshape(X.shape[0], -1)
    def inverse_transform(self, X): 
        return X.reshape(-1, 72, len(time_series_columns))

# Enhanced Data Preprocessing
def preprocess_data(df):
    time_series_columns = [col for col in df.columns if 'prev_72h_' in col]
    
    # Convert time series data to float32 arrays
    X = np.stack([df[col].apply(safe_eval).values for col in time_series_columns], axis=-1)
    
    # Handle NaNs with temporal-aware imputation
    for f_idx in range(X.shape[-1]):
        feature = X[..., f_idx]
        feature[np.isnan(feature)] = np.nanmean(feature)
    
    # Encode labels as one-hot vectors
    le = LabelEncoder()
    y = to_categorical(le.fit_transform(df['event_type']))
    
    return X.astype('float32'), y, le.classes_

# Build LSTM Model with AUC Compatibility
def build_model(input_shape, num_classes):
    model = Sequential([
        Masking(mask_value=np.nan, input_shape=input_shape),
        Bidirectional(LSTM(128, return_sequences=True, dropout=0.3)),
        Bidirectional(LSTM(64, dropout=0.2)),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Custom AUC Callback for Multi-Class
from tensorflow.keras.callbacks import Callback
from sklearn.metrics import roc_auc_score

class MacroAUC(Callback):
    def __init__(self, X_val, y_val):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val.argmax(axis=1)  # Convert back to integer labels
    
    def on_epoch_end(self, epoch, logs=None):
        y_pred = self.model.predict(self.X_val)
        auc = roc_auc_score(self.y_val, y_pred, multi_class='ovo')
        print(f'\nValidation Macro AUC: {auc:.4f}')

# Main Execution Pipeline
def main(df):
    X, y, classes = preprocess_data(df)
    num_classes = len(classes)
    
    tscv = TimeSeriesSplit(n_splits=5)
    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_val = X[train_idx], X[test_idx]
        y_train, y_val = y[train_idx], y[test_idx]
        
        # Class balancing
        sample_weights = compute_class_weight('balanced', classes=np.unique(y_train.argmax(1)), y=y_train.argmax(1))
        sample_weights = sample_weights[y_train.argmax(1)]
        
        model = build_model(X.shape[1:], num_classes)
        auc_callback = MacroAUC(X_val, y_val)
        
        model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=100,
            batch_size=64,
            sample_weight=sample_weights,
            callbacks=[auc_callback],
            verbose=1
        )
    
    return model

# Execute the corrected pipeline
final_model = main(df)


TypeError: ufunc 'divide' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [ ]:
X.shape

(5491,)

In [59]:
%pip install imblearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 4.6 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip available: 22.2.2 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
